# Carteira Polo Parnaiba - Tratamento e Analise Exploratoria

Base mensal de clientes (cadastro, transacional e receita) do Polo Franquia de Parnaiba.
Este notebook trata a base e constroi os graficos que alimentarao o dashboard interativo.

## 1. Importacao e consolidacao por Documento


In [18]:
# =============================================================================
# 1. IMPORTACAO DA BASE E CONSOLIDACAO POR DOCUMENTO
# =============================================================================
# A base bruta traz uma linha por (mes de referencia, Stonecode). Como um mesmo
# cliente (Documento) pode ter varios Stonecodes - tipicamente um cadastro
# principal + cadastros acessorios de Link ABC / WhatsAppPay / Tap On Phone -,
# consolidamos tudo em UMA linha por (data_referencia, Documento):
#   - metricas transacionais / receita  -> SOMA dos Stonecodes
#   - flags                             -> MAX (basta um cadastro possuir)
#   - datas de inicio -> MIN | datas de fim -> MAX
#   - duration medio                    -> media ponderada pelo TPV de credito
#   - variaveis categoricas             -> herdadas do Stonecode PRINCIPAL

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# O notebook roda tanto aberto no Jupyter (diretorio de trabalho = notebooks/)
# quanto executado por dentro dos scripts de build (diretorio = raiz). Subimos a
# arvore ate achar a raiz do repositorio e importamos os caminhos oficiais de la.
_RAIZ = next(pasta for pasta in [Path.cwd(), *Path.cwd().parents]
             if (pasta / "data").is_dir() and (pasta / "notebooks").is_dir())
if str(_RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(_RAIZ / "src"))
import caminhos

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

CAMINHO_BASE = caminhos.BASE_PARQUET

df_raw = pd.read_parquet(CAMINHO_BASE)
df_raw["data_referencia"] = pd.to_datetime(df_raw["data_referencia"])

print(f"Base bruta: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
print(f"Periodo   : {df_raw['data_referencia'].min():%Y-%m} a {df_raw['data_referencia'].max():%Y-%m} "
      f"({df_raw['data_referencia'].nunique()} meses)")
print(f"Documentos: {df_raw['Documento'].nunique():,} | Stonecodes: {df_raw['Stonecode'].nunique():,}")

# -----------------------------------------------------------------------------
# 1.1 Eleicao do Stonecode principal
# -----------------------------------------------------------------------------
# Canal_venda vem com grafias inconsistentes (Franquia/FRANQUIA, Polo/POLO
# PROPRIO...). Normalizamos apenas para aplicar a regra de escolha; a coluna
# original e preservada no resultado.
canal_norm = (
    df_raw["Canal_venda"].fillna("")
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
    .str.upper().str.replace(r"[^A-Z]", "", regex=True)
)

# Cadastros acessorios: existem para viabilizar um meio de venda especifico e
# nao representam a classificacao comercial do cliente.
CANAIS_ACESSORIOS = {"LINKABC", "WHATSAPPPAY", "TAPONPHONE"}

df_raw["_canal_norm"] = canal_norm
df_raw["_acessorio"] = canal_norm.isin(CANAIS_ACESSORIOS).astype(int)
df_raw["_ativo_m0"] = (df_raw["status_ba_m0"].str.strip().str.casefold() == "ativo").astype(int)
df_raw["_ativo_m1"] = (df_raw["status_ba_m1"].str.strip().str.casefold() == "ativo").astype(int)
df_raw["_tpv_rank"] = df_raw["tpv_m0"].fillna(0)
df_raw["_equip_rank"] = df_raw["Qtd_equipamentos"].fillna(0)

CHAVE = ["data_referencia", "Documento"]

# Prioridade: canal nao acessorio > ativo no mes > maior TPV > mais equipamentos
# > credenciamento mais antigo > Stonecode (desempate estavel).
ordem = df_raw.sort_values(
    by=CHAVE + ["_acessorio", "_ativo_m0", "_tpv_rank", "_equip_rank",
                "Data_credenciamento", "Stonecode"],
    ascending=[True, True, True, False, False, False, True, True],
    kind="mergesort",
)
principal = ordem.drop_duplicates(subset=CHAVE, keep="first")

# -----------------------------------------------------------------------------
# 1.2 Grupos de colunas
# -----------------------------------------------------------------------------
COLS_TPV_DIARIO = [f"tpv_d{d}" for d in range(1, 32)]

COLS_SOMA = (
    ["TPV_estimado", "TPV_esperado_contrato"]
    + COLS_TPV_DIARIO
    + ["tpv_m0", "tpv_m1", "tpv_m2", "tpv_m3",
       "tpv_credito_m0", "tpv_debito_m0", "transacoes",
       "tpv_antecipado_m0", "tpv_antecipado_auto_m0", "tpv_antecipado_spot_m0",
       "tpv_antecipado_m1", "tpv_antecipado_m2", "tpv_antecipado_m3",
       "dx_m0", "Mensalidade_m0", "Qtd_equipamentos",
       "Receita_Bruta_Banking", "Receita_Liquida_Banking",
       "TPV_Cartao", "Receita_interchange",
       "TPV_pix_in_qr_code", "Receita_pix_in_qr_code",
       "Receita_Invoice", "Receita_boleto", "Receita_GDA",
       "Receita_saques", "Receita_recargas",
       "Seguro_Vida_m0", "Seguro_Loja_m0"]
)

COLS_MAX_FLAG = ["Tem_seguro", "Flag_IPV"]      # basta um cadastro possuir
COLS_MIN_NUM = ["dias_sem_transacionar"]        # o cadastro mais recente manda
COLS_DATA_MIN = ["Data_credenciamento", "Data_primeira_ativacao",
                 "Data_primeira_transacao", "Data_assinatura_contrato"]
COLS_DATA_MAX = ["Data_ultima_transacao", "Data_fechamento_conta_stone"]

COLS_CATEGORICAS = ["Tipo_documento", "Nome_fantasia", "Status", "Cidade", "UF",
                    "Vendedor", "Canal_venda", "Cadastro_RAV", "Mcc_key", "mcc",
                    "Domicilio_bancario", "Tipo_contrato", "Rota"]

# -----------------------------------------------------------------------------
# 1.3 Agregacao
# -----------------------------------------------------------------------------
g = df_raw.groupby(CHAVE, sort=True)

# min_count=1 preserva NaN quando TODOS os Stonecodes do documento sao nulos,
# evitando transformar ausencia de informacao em zero.
agg_soma = g[COLS_SOMA].sum(min_count=1)
agg_max = g[COLS_MAX_FLAG].max()
agg_min = g[COLS_MIN_NUM].min()
agg_dmin = g[COLS_DATA_MIN].min()
agg_dmax = g[COLS_DATA_MAX].max()

# duration_m0: media ponderada pelo TPV de credito (fallback TPV total, depois media
# simples). Pesos negativos (estornos) sao zerados para nao inverter a ponderacao.
peso = df_raw["tpv_credito_m0"].fillna(0).clip(lower=0)
peso = peso.where(peso > 0, df_raw["tpv_m0"].fillna(0).clip(lower=0))
df_raw["_dur_num"] = df_raw["duration_m0"].fillna(0) * peso
df_raw["_dur_den"] = peso.where(df_raw["duration_m0"].notna(), 0)
dur = g[["_dur_num", "_dur_den"]].sum()
duration_pond = (dur["_dur_num"] / dur["_dur_den"].replace(0, np.nan))
duration_pond = duration_pond.fillna(g["duration_m0"].mean()).rename("duration_m0")

# Estrutura de cadastros do documento
agg_estrutura = g.agg(
    qtd_stonecodes=("Stonecode", "size"),
    qtd_stonecodes_ativos=("_ativo_m0", "sum"),
    ativo_m0=("_ativo_m0", "max"),
    ativo_m1=("_ativo_m1", "max"),
)
canais_doc = (g["_canal_norm"]
              .agg(lambda s: "|".join(sorted({v for v in s if v})))
              .rename("canais_venda"))

bloco_principal = (
    principal.set_index(CHAVE)[["Stonecode"] + COLS_CATEGORICAS]
    .rename(columns={"Stonecode": "Stonecode_principal"})
)

df = pd.concat(
    [bloco_principal, agg_estrutura, canais_doc, agg_soma, agg_max, agg_min,
     duration_pond, agg_dmin, agg_dmax],
    axis=1,
).reset_index()

# status_ba consolidado: o cliente e ativo se QUALQUER cadastro dele transacionou
df["status_ba_m0"] = np.where(df["ativo_m0"] == 1, "Ativo", "Inativo")
df["status_ba_m1"] = np.where(df["ativo_m1"] == 1, "Ativo", "Inativo")
df["flag_multi_stonecode"] = (df["qtd_stonecodes"] > 1).astype(int)
df = df.drop(columns=["ativo_m0", "ativo_m1"])

df_raw = df_raw.drop(columns=[c for c in df_raw.columns if c.startswith("_")])

# -----------------------------------------------------------------------------
# 1.4 Descarte de variaveis descontinuadas
# -----------------------------------------------------------------------------
# Colunas que mudam de definicao ou deixam de ser alimentadas no meio da serie
# nao servem para modelagem preditiva: o modelo aprende um regime que nao existe
# mais. Sao removidas da base de trabalho e permanecem em df_raw para auditoria.
COLS_DESCONTINUADAS = {
    "Receita_GDA": "zerada a partir de jan/2026",
    "Receita_saques": "zerada a partir de jan/2026",
    "Receita_recargas": "zerada a partir de jan/2026",
    "Receita_Invoice": "zerada a partir de jan/2026",
    "TPV_Cartao": "pico isolado em jan/2026 e zerada a partir de fev/2026",
    "Receita_boleto": "redefinida em jan/2026 (de ~80 para ~700 cadastros por mes)",
}

_por_mes = df_raw.groupby(df_raw["data_referencia"].dt.to_period("M"))
_positivos = _por_mes[list(COLS_DESCONTINUADAS)].apply(lambda x: (x.fillna(0) > 0).sum())

print("\nVariaveis descartadas por descontinuidade na serie:")
for col, motivo in COLS_DESCONTINUADAS.items():
    meses_com_dado = _positivos[col][_positivos[col] > 0]
    print(f"  {col:<22} {motivo:<58} "
          f"{len(meses_com_dado)} de {len(_positivos)} meses com dado, "
          f"ultimo em {meses_com_dado.index.max()}")

# Detector automatico: alerta se alguma outra coluna numerica secar no fim da serie
_num = [c for c in df_raw.columns
        if df_raw[c].dtype.kind in "fi" and c not in COLS_DESCONTINUADAS]
_share = _por_mes[_num].apply(lambda x: (x.fillna(0) > 0).mean())
_secas = [c for c in _num if (_share[c].tail(6) <= 0.0005).all() and _share[c].head(12).max() > 0.01]
print(f"  Outras colunas secas nos ultimos 6 meses: {_secas if _secas else 'nenhuma'}")

df = df.drop(columns=[c for c in COLS_DESCONTINUADAS if c in df.columns])

# Receita_interchange segue alimentada (~130 a 150 cadastros/mes) mesmo sem
# TPV_Cartao, entao foi mantida - a receita passou a existir sem o volume.

# -----------------------------------------------------------------------------
# 1.5 Conferencias
# -----------------------------------------------------------------------------
print(f"\nBase consolidada: {df.shape[0]:,} linhas x {df.shape[1]} colunas "
      f"({1 - df.shape[0] / df_raw.shape[0]:.1%} menos linhas)")
print(f"Chave (data_referencia, Documento) unica: {not df.duplicated(CHAVE).any()}")

print("\nSomas preservadas apos a consolidacao:")
for col in ["tpv_m0", "tpv_credito_m0", "transacoes", "Receita_Liquida_Banking",
            "Mensalidade_m0", "Qtd_equipamentos"]:
    dif = df[col].sum() - df_raw[col].sum()
    print(f"  {col:<24} diferenca = {dif:,.4f}")

print(f"\nLinhas de documentos com mais de um Stonecode: {df['flag_multi_stonecode'].mean():.1%}")
print(f"Clientes ativos no mes (status_ba_m0): {(df['status_ba_m0'] == 'Ativo').mean():.1%}")

df.head(5)

Base bruta: 240,847 linhas x 90 colunas
Periodo   : 2024-01 a 2026-07 (31 meses)
Documentos: 7,076 | Stonecodes: 13,980

Variaveis descartadas por descontinuidade na serie:
  Receita_GDA            zerada a partir de jan/2026                                24 de 31 meses com dado, ultimo em 2025-12
  Receita_saques         zerada a partir de jan/2026                                24 de 31 meses com dado, ultimo em 2025-12
  Receita_recargas       zerada a partir de jan/2026                                23 de 31 meses com dado, ultimo em 2025-12
  Receita_Invoice        zerada a partir de jan/2026                                24 de 31 meses com dado, ultimo em 2025-12
  TPV_Cartao             pico isolado em jan/2026 e zerada a partir de fev/2026     25 de 31 meses com dado, ultimo em 2026-01
  Receita_boleto         redefinida em jan/2026 (de ~80 para ~700 cadastros por mes) 29 de 31 meses com dado, ultimo em 2026-07
  Outras colunas secas nos ultimos 6 meses: nenhuma

Base consol

,data_referencia,Documento,Stonecode_principal,Tipo_documento,Nome_fantasia,Status,Cidade,UF,Vendedor,Canal_venda,Cadastro_RAV,Mcc_key,mcc,Domicilio_bancario,Tipo_contrato,Rota,qtd_stonecodes,qtd_stonecodes_ativos,canais_venda,TPV_estimado,TPV_esperado_contrato,tpv_d1,tpv_d2,tpv_d3,tpv_d4,tpv_d5,tpv_d6,tpv_d7,tpv_d8,tpv_d9,tpv_d10,tpv_d11,tpv_d12,tpv_d13,tpv_d14,tpv_d15,tpv_d16,tpv_d17,tpv_d18,tpv_d19,tpv_d20,tpv_d21,tpv_d22,tpv_d23,tpv_d24,tpv_d25,tpv_d26,tpv_d27,tpv_d28,tpv_d29,tpv_d30,tpv_d31,tpv_m0,tpv_m1,tpv_m2,tpv_m3,tpv_credito_m0,tpv_debito_m0,transacoes,tpv_antecipado_m0,tpv_antecipado_auto_m0,tpv_antecipado_spot_m0,tpv_antecipado_m1,tpv_antecipado_m2,tpv_antecipado_m3,dx_m0,Mensalidade_m0,Qtd_equipamentos,Receita_Bruta_Banking,Receita_Liquida_Banking,Receita_interchange,TPV_pix_in_qr_code,Receita_pix_in_qr_code,Seguro_Vida_m0,Seguro_Loja_m0,Tem_seguro,Flag_IPV,dias_sem_transacionar,duration_m0,Data_credenciamento,Data_primeira_ativacao,Data_primeira_transacao,Data_assinatura_contrato,Data_ultima_transacao,Data_fechamento_conta_stone,status_ba_m0,status_ba_m1,flag_multi_stonecode
0,2024-01-31,cnpj_10,stonecode_10250,CNPJ,cliente_3859,Cancelado,PARNAIBA,PI,colaborador_235,Franquia,Fast,5199,A/D DE MERCADORIAS NÃO DURÁVEIS (NÃO CLASSIF. ...,Itaú Unibanco S.A.,NaN,Parnaíba Centro,1,0,FRANQUIA,"15,000.00",NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,1.00,470.00,0.00,2020-11-23,2020-11-24,2020-11-24,NaT,2022-11-01,NaT,Inativo,Inativo,0
1,2024-01-31,cnpj_100,stonecode_3693,CNPJ,cliente_3949,Aprovado - Pronto para transacionar,PARNAIBA,PI,colaborador_158,Franquia,Auto,8351,SERVIÇOS DE CUIDADOS DE CRIANÇAS (CHILD CARE ...,Stone Pagamentos S.A.,NaN,Parnaíba Centro,1,0,FRANQUIA,"5,000.00",NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,NaN,0.00,2023-09-27,NaT,NaT,NaT,NaT,NaT,Inativo,Inativo,0
2,2024-01-31,cnpj_1001,stonecode_12047,MEI,cliente_4850,Aprovado - Pronto para transacionar,PARNAIBA,PI,NaN,Link ABC,Spot,4722,AGÊNCIAS DE VIAGENS (TRAVEL AGENCIES),Stone Pagamentos S.A.,NaN,Parnaíba Mid,1,0,LINKABC,"10,000.00",NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,NaN,NaN,0.00,2022-06-16,NaT,NaT,NaT,NaT,NaT,Inativo,Inativo,0
3,2024-01-31,cnpj_1002,stonecode_13498,CNPJ,cliente_4851,Cancelado,PARNAIBA,PI,colaborador_75,Comercial,Spot,5571,LOJAS DE MOTOCICLETAS E ACESSÓRIOS,Banco Santander (Brasil) S.A.,NaN,Parnaíba Mid,1,0,COMERCIAL,"15,000.00",NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,"1,350.00",0.00,2020-01-20,2020-01-22,2020-01-22,NaT,2020-06-04,NaT,Inativo,Inativo,0
4,2024-01-31,cnpj_1004,stonecode_3786,MEI,cliente_4853,Aprovado - Pronto para transacionar,PARNAIBA,PI,colaborador_327,Franquia,Auto,5812,RESTAURANTES,Stone Pagamentos S.A.,NaN,Parnaíba Mid,2,0,FRANQUIA|LINKABC,"30,000.00",NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0.00,691.00,0.00,2022-01-24,2022-01-24,2022-01-24,NaT,2022-03-25,NaT,Inati

## 2. Feature engineering

Celula central de criacao de variaveis do projeto. Toda feature nova entra aqui.

O PIX QR Code passa por validacao: so entra no TPV Total quando o documento tem
equipamento instalado no mes, porque o QR e gerado na POS.

In [19]:
# =============================================================================
# 2. FEATURE ENGINEERING
# =============================================================================
# Celula central de criacao de variaveis do projeto: toda feature nova entra aqui.
#
# 2.1 TPV Total  = tpv_m0 + PIX QR Code VALIDADO (adquirencia + PIX na POS)
# 2.2 Flags de fluxo mensal da carteira, a partir do historico de TPV Total:
#       Novo Ativo  -> transacionou no mes e estava zerado nos 3 meses anteriores
#       Reativacao  -> transacionou no mes, zerado no mes anterior, mas com TPV
#                      em M-2 ou M-3 (ou seja, nao e novo ativo)
#       Churn       -> zerado no mes tendo transacionado no mes anterior
# 2.3 Qualidade da venda nova e da saida: flag_novo_ativo_m1, tpv_total_medio_m1_m3
# 2.4 Rota_atual, a rota vigente do cliente propagada para todo o historico
#
# Convencoes:
#   - "transacionou" = TPV Total > 0; "zerado" = TPV Total <= 0 (11 linhas da base
#     tem TPV liquido negativo por estorno e sao tratadas como sem transacao).
#   - TRATAMENTO DO PIX: o PIX QR Code so e considerado quando o documento tem
#     equipamento instalado no mes (o QR e gerado na POS). Ate ago/2025 a base
#     reportava PIX para cadastros dormentes sem POS - volume que sumiu de uma vez
#     em set/2025 e inflava artificialmente a base ativa e o churn do mes seguinte.
#   - o historico e lido em um painel DENSO (Documento x todos os meses): se o
#     documento nao aparece em um mes, seu TPV naquele mes e zero.
#   - Novo Ativo e Reativacao sao mutuamente exclusivos e, somados, cobrem todo
#     cliente que voltou a transacionar apos um mes zerado.

# -----------------------------------------------------------------------------
# 2.1 Validacao do PIX QR Code e TPV Total
# -----------------------------------------------------------------------------
# Colunas originais da base sao preservadas; o tratamento gera colunas novas.
equipamentos = df["Qtd_equipamentos"].fillna(0)
pix_bruto = df["TPV_pix_in_qr_code"].fillna(0)
tem_equipamento = equipamentos > 0

df["flag_pix_sem_equipamento"] = (pix_bruto.gt(0) & ~tem_equipamento).astype("int8")
df["pix_qr"] = pix_bruto.where(tem_equipamento, 0.0)            # PIX considerado
df["pix_qr_descartado"] = pix_bruto - df["pix_qr"]              # PIX removido
# a receita acompanha o mesmo criterio, para nao distorcer o take rate do PIX
df["receita_pix_qr"] = df["Receita_pix_in_qr_code"].fillna(0).where(tem_equipamento, 0.0)

df["tpv_total"] = df["tpv_m0"].fillna(0) + df["pix_qr"]
df["transacionou"] = (df["tpv_total"] > 0).astype("int8")

_pix_total = pix_bruto.sum()
_descartado = df["pix_qr_descartado"].sum()
print(f"PIX QR Code descartado (documento sem equipamento no mes): "
      f"R$ {_descartado:,.2f} ({_descartado / _pix_total:.2%} do PIX, "
      f"{_descartado / (df['tpv_total'].sum() + _descartado):.2%} do TPV Total)")
print(f"Linhas afetadas: {df['flag_pix_sem_equipamento'].sum():,} "
      f"(das quais {int((df['flag_pix_sem_equipamento'].eq(1) & df['tpv_m0'].fillna(0).gt(0)).sum()):,} "
      f"seguem ativas por TPV de adquirencia)")

# -----------------------------------------------------------------------------
# 2.2 Historico de TPV Total em painel denso (Documento x mes)
# -----------------------------------------------------------------------------
df["periodo"] = df["data_referencia"].dt.to_period("M")

meses = pd.period_range(df["periodo"].min(), df["periodo"].max(), freq="M")
painel = (df.pivot_table(index="Documento", columns="periodo", values="tpv_total",
                         aggfunc="sum")
            .reindex(columns=meses))
presente = painel.notna()          # o documento existia na base naquele mes
painel = painel.fillna(0.0)        # ausencia da linha = nao transacionou

lag1, lag2, lag3 = (painel.shift(k, axis=1) for k in (1, 2, 3))

tx_m0 = painel > 0
tx_m1, tx_m2, tx_m3 = lag1 > 0, lag2 > 0, lag3 > 0

# Nos primeiros meses do painel nao ha historico suficiente: as janelas ficam
# indefinidas (NaN nos lags) e as flags nao sao calculadas.
hist1, hist3 = lag1.notna(), lag3.notna()

m_novo_ativo = tx_m0 & ~tx_m1 & ~tx_m2 & ~tx_m3 & hist3
m_reativacao = tx_m0 & ~tx_m1 & hist1 & ~m_novo_ativo
m_churn = ~tx_m0 & tx_m1 & hist1

# Quantos eventos caem em meses em que o documento nao existe mais na base
# (cadastros que sairam do escopo da carteira e nao tem linha para receber a flag)
eventos_fora = int((m_churn & ~presente).values.sum())

def _para_long(matriz, nome):
    return (matriz.stack().rename(nome).astype("int8")
            .rename_axis(["Documento", "periodo"]))

flags = pd.concat(
    [_para_long(m_novo_ativo, "flag_novo_ativo"),
     _para_long(m_reativacao, "flag_reativacao"),
     _para_long(m_churn, "flag_churn")],
    axis=1,
)

hist_lags = pd.concat(
    [lag1.stack().rename("tpv_total_m1"),
     lag2.stack().rename("tpv_total_m2"),
     lag3.stack().rename("tpv_total_m3")],
    axis=1,
).rename_axis(["Documento", "periodo"])

df = (df.merge(hist_lags, how="left", left_on=["Documento", "periodo"], right_index=True)
        .merge(flags, how="left", left_on=["Documento", "periodo"], right_index=True))

for col in ["flag_novo_ativo", "flag_reativacao", "flag_churn"]:
    df[col] = df[col].fillna(0).astype("int8")

# Janela de historico disponivel: mes com 3 meses anteriores dentro do painel
df["janela_hist_completa"] = (df["periodo"] >= meses[0] + 3).astype("int8")

# Status de movimentacao consolidado em uma unica variavel categorica
df["status_movimento"] = np.select(
    [df["flag_novo_ativo"] == 1, df["flag_reativacao"] == 1, df["flag_churn"] == 1,
     df["transacionou"] == 1],
    ["Novo Ativo", "Reativacao", "Churn", "Ativo recorrente"],
    default="Sem transacao",
)

# -----------------------------------------------------------------------------
# 2.3 Qualidade da venda nova e da saida
# -----------------------------------------------------------------------------
# Duas features de linha para medir QUANTO cada entrada traz e quanto cada saida
# leva, em vez de so contar clientes:
#   flag_novo_ativo_m1     -> o documento foi Novo Ativo no mes ANTERIOR. Cruzada
#                             com tpv_total, mede o TPV do primeiro mes cheio de
#                             quem acabou de entrar.
#   tpv_total_medio_m1_m3  -> TPV Total medio dos 3 meses anteriores. Cruzada com
#                             flag_churn, mede o tamanho do cliente perdido antes
#                             da queda que antecede a saida.

_flag_anterior = (df[["Documento", "periodo", "flag_novo_ativo"]]
                  .rename(columns={"flag_novo_ativo": "flag_novo_ativo_m1"}))
_flag_anterior["periodo"] = _flag_anterior["periodo"] + 1   # vale para o mes seguinte
df = df.merge(_flag_anterior, on=["Documento", "periodo"], how="left")
df["flag_novo_ativo_m1"] = df["flag_novo_ativo_m1"].fillna(0).astype("int8")

# media simples dos 3 lags; fica nula so quando nenhum dos tres existe
df["tpv_total_medio_m1_m3"] = df[["tpv_total_m1", "tpv_total_m2",
                                  "tpv_total_m3"]].mean(axis=1)

# -----------------------------------------------------------------------------
# 2.4 Rota_atual: a rota vigente do cliente propagada para todos os meses
# -----------------------------------------------------------------------------
# A malha de rotas foi redesenhada ao longo da serie: "Parnaiba Mid" e "Parnaiba
# Norte" deixaram de existir, "Rota Litoral PI" virou "Area Litoral PI" e parte
# dos clientes trocou de rota sem trocar de endereco. Comparar rota mes a mes
# misturaria mudanca de carteira com mudanca de regua. Rota_atual fixa a
# classificacao mais recente de CADA documento e a aplica a todo o historico,
# permitindo acompanhar a evolucao dos indicadores por regiao geografica.

def _chave_rota(serie):
    """Normaliza a grafia (caixa, acento, underscore) para comparar rotas."""
    return (serie.astype("string")
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
            .str.replace(r"[_\s]+", " ", regex=True)
            .str.strip().str.upper())

df["_rota_chave"] = _chave_rota(df["Rota"])

# Buckets genericos de "outras rotas" aparecem com grafias diferentes por epoca
# (Outras Rotas / Rota_Outros__C / Rota_Outros__c) e sao unificados.
CHAVES_OUTRAS = {"OUTRAS ROTAS", "ROTA OUTROS C"}
df.loc[df["_rota_chave"].isin(CHAVES_OUTRAS), "_rota_chave"] = "OUTRAS ROTAS"

# Rotulo de exibicao: a grafia usada no mes mais recente em que a rota aparece
rotulo = (df.dropna(subset=["_rota_chave"])
            .sort_values("periodo")
            .groupby("_rota_chave")["Rota"].last())
rotulo["OUTRAS ROTAS"] = "Outras Rotas"

# Ultima classificacao conhecida de cada documento
ultima_rota = (df.dropna(subset=["_rota_chave"])
                 .sort_values("periodo")
                 .groupby("Documento")
                 .agg(_chave=("_rota_chave", "last"), rota_atual_mes=("periodo", "last")))
ultima_rota["Rota_atual"] = ultima_rota["_chave"].map(rotulo)

df = df.merge(ultima_rota[["Rota_atual", "rota_atual_mes"]], how="left",
              left_on="Documento", right_index=True)
df["Rota_atual"] = df["Rota_atual"].fillna("Nao informada")
df = df.drop(columns=["_rota_chave"])

# -----------------------------------------------------------------------------
# 2.5 Conferencias
# -----------------------------------------------------------------------------
print(f"TPV Total criado: soma = R$ {df['tpv_total'].sum():,.2f} "
      f"({df['transacionou'].sum():,} linhas com transacao)")
print(f"Participacao do PIX QR Code validado no TPV Total: "
      f"{df['pix_qr'].sum() / df['tpv_total'].sum():.2%}")
print(f"\nNovo Ativo e Reativacao se sobrepoem? "
      f"{bool((df['flag_novo_ativo'] & df['flag_reativacao']).any())}")
print(f"Churn sobrepoe alguma flag de entrada? "
      f"{bool((df['flag_churn'] & (df['flag_novo_ativo'] | df['flag_reativacao'])).any())}")
print(f"Eventos de churn em meses sem linha na base (fora do escopo): {eventos_fora}")
print(f"Meses sem janela de 3 periodos (flags nao avaliadas): "
      f"{sorted(df.loc[df['janela_hist_completa'] == 0, 'periodo'].unique())}")

resumo_fluxo = (df[df["janela_hist_completa"] == 1]
                .groupby("periodo")
                .agg(base_ativa=("transacionou", "sum"),
                     novos_ativos=("flag_novo_ativo", "sum"),
                     reativacoes=("flag_reativacao", "sum"),
                     churn=("flag_churn", "sum"),
                     tpv_total=("tpv_total", "sum")))
resumo_fluxo["saldo"] = (resumo_fluxo["novos_ativos"] + resumo_fluxo["reativacoes"]
                         - resumo_fluxo["churn"])
resumo_fluxo["churn_pct_base_m1"] = (resumo_fluxo["churn"]
                                     / resumo_fluxo["base_ativa"].shift(1) * 100).round(1)

print("\nFluxo mensal da carteira (ultimos 12 meses):")
print(resumo_fluxo.tail(12).to_string())

# O PIX descartado se concentra ate ago/2025, exatamente o periodo em que a base
# reportava PIX para cadastros dormentes sem POS.
pix_descartado_mes = (df.groupby("periodo")
                        .agg(docs_sem_equip=("flag_pix_sem_equipamento", "sum"),
                             pix_descartado=("pix_qr_descartado", "sum")))
print("\nPIX descartado por mes (jan/2025 em diante):")
print(pix_descartado_mes.loc["2025-01":].to_string())

print("\nDistribuicao de status_movimento:")
print(df["status_movimento"].value_counts().to_string())

docs_rota = df.drop_duplicates("Documento")
print(f"\nRota_atual definida para {(docs_rota['Rota_atual'] != 'Nao informada').sum():,} "
      f"de {len(docs_rota):,} documentos")
print("Documentos por Rota_atual (mes de origem da classificacao):")
print(docs_rota.groupby("Rota_atual")
               .agg(documentos=("Documento", "size"),
                    mes_origem_min=("rota_atual_mes", "min"),
                    mes_origem_max=("rota_atual_mes", "max"))
               .sort_values("documentos", ascending=False).to_string())

_divergente = df["Rota"].notna() & (_chave_rota(df["Rota"]).replace(
    {k: "OUTRAS ROTAS" for k in CHAVES_OUTRAS}) != _chave_rota(df["Rota_atual"]))
print(f"\nLinhas em que a rota do mes difere da Rota_atual: {_divergente.mean():.1%} "
      "(redesenho da malha ao longo da serie)")

print("\nTPV Total por Rota_atual e ano (R$ milhoes):")
print((df.assign(ano=df["data_referencia"].dt.year)
         .pivot_table(index="Rota_atual", columns="ano", values="tpv_total",
                      aggfunc="sum")
         .div(1e6).round(1).to_string()))

df[["data_referencia", "Documento", "Rota", "Rota_atual", "Qtd_equipamentos",
    "tpv_m0", "TPV_pix_in_qr_code", "pix_qr", "pix_qr_descartado", "tpv_total",
    "tpv_total_m1", "tpv_total_m2", "tpv_total_m3", "flag_novo_ativo",
    "flag_reativacao", "flag_churn", "status_movimento"]].head(5)

PIX QR Code descartado (documento sem equipamento no mes): R$ 11,990,503.66 (4.45% do PIX, 0.95% do TPV Total)
Linhas afetadas: 2,849 (das quais 353 seguem ativas por TPV de adquirencia)
TPV Total criado: soma = R$ 1,253,516,804.74 (45,775 linhas com transacao)
Participacao do PIX QR Code validado no TPV Total: 20.56%

Novo Ativo e Reativacao se sobrepoem? False
Churn sobrepoe alguma flag de entrada? False
Eventos de churn em meses sem linha na base (fora do escopo): 36
Meses sem janela de 3 periodos (flags nao avaliadas): [Period('2024-01', 'M'), Period('2024-02', 'M'), Period('2024-03', 'M')]

Fluxo mensal da carteira (ultimos 12 meses):
         base_ativa  novos_ativos  reativacoes  churn     tpv_total  saldo  churn_pct_base_m1
periodo                                                                                      
2025-08        1502            52           19     76 44,359,572.98     -5               5.00
2025-09        1507            60           20     73 39,443,925.17   

,data_referencia,Documento,Rota,Rota_atual,Qtd_equipamentos,tpv_m0,TPV_pix_in_qr_code,pix_qr,pix_qr_descartado,tpv_total,tpv_total_m1,tpv_total_m2,tpv_total_m3,flag_novo_ativo,flag_reativacao,flag_churn,status_movimento
0,2024-01-31,cnpj_10,Parnaíba Centro,Parnaíba Centro,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0,0,0,Sem transacao
1,2024-01-31,cnpj_100,Parnaíba Centro,Parnaíba Centro,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0,0,0,Sem transacao
2,2024-01-31,cnpj_1001,Parnaíba Mid,Parnaíba Sul,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0,0,0,Sem transacao
3,2024-01-31,cnpj_1002,Parnaíba Mid,Parnaíba Sul,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0,0,0,Sem transacao
4,2024-01-31,cnpj_1004,Parnaíba Mid,Parnaíba Centro,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN,NaN,0,0,0,Sem transacao


## 3. Backup da base tratada

Exporta `base_edited.parquet` com todo o tratamento aplicado. E apenas backup: o
projeto segue partindo de `base.parquet` e refazendo o tratamento nas celulas acima.

In [20]:
# =============================================================================
# 3. BACKUP DA BASE TRATADA
# =============================================================================
# Exporta o resultado de todo o tratamento (consolidacao por documento, descarte
# de variaveis descontinuadas, validacao do PIX, flags de fluxo e Rota_atual).
#
# ATENCAO: arquivo de BACKUP. O projeto continua partindo de data/raw/base.parquet e
# refazendo o tratamento nas celulas acima, para que qualquer mudanca de regra
# se propague. Este parquet nao e lido em nenhuma etapa seguinte.

CAMINHO_BACKUP = caminhos.BASE_EDITADA_PARQUET

# Colunas Period nao tem suporte em parquet: viram texto no formato AAAA-MM.
backup = df.copy()
for col in backup.columns:
    if isinstance(backup[col].dtype, pd.PeriodDtype):
        backup[col] = backup[col].astype("string")

backup.to_parquet(CAMINHO_BACKUP, index=False, compression="snappy")

# Conferencia: le de volta e compara com o dataframe em memoria
conferencia = pd.read_parquet(CAMINHO_BACKUP)
tamanho_mb = CAMINHO_BACKUP.stat().st_size / 1024**2

print(f"Arquivo gravado: {CAMINHO_BACKUP.resolve()}")
print(f"Tamanho: {tamanho_mb:,.1f} MB "
      f"(base.parquet original: {CAMINHO_BASE.stat().st_size / 1024**2:,.1f} MB)")
print(f"Formato: {conferencia.shape[0]:,} linhas x {conferencia.shape[1]} colunas")
print(f"Linhas e colunas conferem com o dataframe tratado: "
      f"{conferencia.shape == backup.shape}")
print(f"TPV Total conferido na releitura: R$ {conferencia['tpv_total'].sum():,.2f} "
      f"(diferenca = {conferencia['tpv_total'].sum() - df['tpv_total'].sum():,.4f})")
print(f"Periodo coberto: {conferencia['data_referencia'].min():%Y-%m} a "
      f"{conferencia['data_referencia'].max():%Y-%m}")
print("\nVariaveis criadas no tratamento presentes no backup:")
_criadas = ["tpv_total", "pix_qr", "pix_qr_descartado", "receita_pix_qr",
            "flag_pix_sem_equipamento", "flag_novo_ativo", "flag_reativacao",
            "flag_churn", "status_movimento", "Rota_atual", "rota_atual_mes",
            "qtd_stonecodes", "canais_venda", "flag_multi_stonecode"]
print("  " + ", ".join(c for c in _criadas if c in conferencia.columns))

Arquivo gravado: C:\DEV\PADS\Integradora\integradora_v1\projeto_churn\data\processed\base_edited.parquet
Tamanho: 18.8 MB (base.parquet original: 14.5 MB)
Formato: 176,514 linhas x 107 colunas
Linhas e colunas conferem com o dataframe tratado: True
TPV Total conferido na releitura: R$ 1,253,516,804.74 (diferenca = 0.0000)
Periodo coberto: 2024-01 a 2026-07

Variaveis criadas no tratamento presentes no backup:
  tpv_total, pix_qr, pix_qr_descartado, receita_pix_qr, flag_pix_sem_equipamento, flag_novo_ativo, flag_reativacao, flag_churn, status_movimento, Rota_atual, rota_atual_mes, qtd_stonecodes, canais_venda, flag_multi_stonecode


## 4. Graficos do dashboard

Cada grafico e uma funcao que recebe o dataframe tratado e devolve uma figura Plotly,
pronta para ser reaproveitada no dashboard interativo.

### 4.1 Evolucao da base ativa

In [21]:
# =============================================================================
# 4. GRAFICOS DO DASHBOARD
# =============================================================================
# Esta celula traz a base comum dos graficos: paletas, series auxiliares e um
# construtor unico de colunas agrupadas por mes com uma coluna por ano. As
# celulas seguintes so chamam esse construtor para cada indicador.
#
# Separadores no padrao brasileiro: milhar com ponto, decimal com virgula.

import numpy as np
import plotly.graph_objects as go

MESES_PT = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
            "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

# Rampas ORDINAIS de um unico hue: mais clara para os anos antigos, mais escura
# para o ano corrente. Cada rampa foi validada contra a superficie do grafico -
# o passo mais claro supera o piso de 2:1 de contraste e os passos vizinhos
# superam dE 15 em visao normal e dE 8 sob simulacao de daltonismo. A tinta do
# rotulo dentro da coluna e escolhida pelo contraste contra cada passo.
PALETAS = {
    "verde": dict(
        light=["#66bf93", "#0f8a58", "#00522a"],
        dark=["#8ad9b0", "#3ba777", "#1a6b47"],
        tinta_light=["#0b0b0b", "#0b0b0b", "#ffffff"],
        tinta_dark=["#0b0b0b", "#0b0b0b", "#ffffff"]),
    "azul": dict(
        light=["#86b6ef", "#256abf", "#0d366b"],
        dark=["#aecdf5", "#3f8ae0", "#18528f"],
        tinta_light=["#0b0b0b", "#ffffff", "#ffffff"],
        tinta_dark=["#0b0b0b", "#0b0b0b", "#ffffff"]),
    "amarelo": dict(
        light=["#dba400", "#a06f00", "#5c3f08"],
        dark=["#f2c75c", "#c98500", "#8a5c00"],
        tinta_light=["#0b0b0b", "#0b0b0b", "#ffffff"],
        tinta_dark=["#0b0b0b", "#0b0b0b", "#ffffff"]),
    "vermelho": dict(
        light=["#ef9393", "#d03b3b", "#7d1c1c"],
        dark=["#f3adad", "#e0504f", "#93292a"],
        tinta_light=["#0b0b0b", "#ffffff", "#ffffff"],
        tinta_dark=["#0b0b0b", "#0b0b0b", "#ffffff"]),
    # verde da bandeira do Brasil (#009c3b) como passo do meio da rampa
    "verde_bandeira": dict(
        light=["#62c586", "#009c3b", "#004d1d"],
        dark=["#8ad9a8", "#2eb35c", "#146e33"],
        tinta_light=["#0b0b0b", "#0b0b0b", "#ffffff"],
        tinta_dark=["#0b0b0b", "#0b0b0b", "#ffffff"]),
}

# Verde bem claro das colunas de fluxo nas pontes: funciona como pano de fundo do
# mes, atras das linhas e dos conectores. Fica em 1,50:1 de contraste contra a
# superficie, abaixo do piso de 2:1 de uma cor que carrega significado sozinha -
# o rotulo em cada coluna e a compensacao que mantem o dado legivel.
VERDE_CLARO = "#a8dcc2"

# Rotulo de variacao: verde quando sobe, vermelho quando cai. Sao os tons de
# delta da paleta, nunca os tons das barras, para o numero nao se confundir com
# a serie. O sinal (+/-) acompanha a cor porque verde e vermelho ficam a dE 5,4
# sob daltonismo e a cor sozinha nao pode carregar o significado.
CHROME = {
    "light": dict(superficie="#fcfcfb", tinta="#0b0b0b", secundaria="#52514e",
                  suave="#898781", grade="#e1e0d9", eixo="#c3c2b7",
                  delta_sobe="#006300", delta_cai="#d03b3b"),
    "dark": dict(superficie="#1a1a19", tinta="#ffffff", secundaria="#c3c2b7",
                 suave="#898781", grade="#2c2c2a", eixo="#383835",
                 delta_sobe="#0ca30c", delta_cai="#e66767"),
}


def _br(valor, casas=0):
    """Formata numero no padrao brasileiro."""
    return f"{valor:,.{casas}f}".replace(",", "@").replace(".", ",").replace("@", ".")


def base_ativa_mensal(dados):
    """Documentos com TPV Total > 0, por mes, como serie continua."""
    ativos = dados.loc[dados["transacionou"] == 1]
    serie = ativos.groupby(ativos["data_referencia"].dt.to_period("M"))["Documento"].nunique()
    return serie.reindex(pd.period_range(serie.index.min(), serie.index.max(), freq="M"))


def matriz_ano_mes(dados, coluna, desde=None, contagem=False):
    """Matriz ano x mes do indicador. Meses sem janela de historico ficam nulos."""
    d = dados if desde is None else dados[dados["periodo"] >= pd.Period(desde, "M")]
    chave = [d["data_referencia"].dt.year, d["data_referencia"].dt.month]
    serie = (d.groupby(chave)[coluna].nunique() if contagem
             else d.groupby(chave)[coluna].sum())
    matriz = serie.unstack().reindex(columns=range(1, 13))
    matriz.index.name = "ano"
    return matriz.reindex(sorted(dados["data_referencia"].dt.year.unique()))


def matriz_taxa(matriz, base_mensal):
    """Indicador dividido pela base ativa do mes ANTERIOR."""
    denominador = pd.DataFrame(index=matriz.index, columns=matriz.columns, dtype=float)
    for ano in matriz.index:
        for mes in matriz.columns:
            anterior = pd.Period(year=ano, month=mes, freq="M") - 1
            denominador.loc[ano, mes] = base_mensal.get(anterior, np.nan)
    return matriz / denominador


def media_ytd(matriz, base_mensal=None):
    """Media mensal de janeiro ate o ultimo mes disponivel do ano mais recente.

    A media - e nao a soma - mantem o acumulado na MESMA escala das colunas
    mensais, o que dispensa um segundo eixo. Anos sem o periodo completo ficam
    nulos, para nao comparar janelas diferentes. A taxa usa a media da base ativa
    dos meses anteriores da mesma janela, ficando comparavel as taxas mensais.
    """
    ultimo_ano = matriz.index.max()
    disponiveis = matriz.loc[ultimo_ano].dropna().index
    janela = list(range(1, int(disponiveis.max()) + 1))
    recorte = matriz[janela]
    media = recorte.mean(axis=1).where(~recorte.isna().any(axis=1))
    taxa = None
    if base_mensal is not None:
        taxa = pd.Series(index=matriz.index, dtype=float)
        for ano in matriz.index:
            anteriores = np.array(
                [base_mensal.get(pd.Period(year=ano, month=m, freq="M") - 1, np.nan)
                 for m in janela], dtype=float)
            taxa[ano] = (media[ano] / anteriores.mean()
                         if not np.isnan(anteriores).any() else np.nan)
    return media, taxa, janela


def grafico_colunas_ano(matriz, paleta, titulo, subtitulo, taxa=None,
                        ytd=None, ytd_taxa=None, rotulo_ytd="Média",
                        alta_e_boa=True, formato=None, modo="light", altura=520):
    """Colunas agrupadas por mes, uma coluna por ano, com grupo opcional de media.

    A media do acumulado do ano fecha o grafico como um 13o grupo, no mesmo eixo
    das colunas mensais - por ser media, e nao soma, fica na mesma ordem de
    grandeza e o grafico segue com uma unica escala.
    """
    anos = list(matriz.index)
    cores = PALETAS[paleta][modo]
    tintas = PALETAS[paleta][f"tinta_{modo}"]
    c = CHROME[modo]
    tem_ytd = ytd is not None

    fig = go.Figure()

    largura, passo = 0.26, 0.28
    desloc = [(i - (len(anos) - 1) / 2) * passo for i in range(len(anos))]
    anotacoes = []

    # A media do acumulado entra como um 13o grupo, no MESMO eixo das colunas
    # mensais, separado por um vao e uma linha tenue. So recebe anos com a janela
    # completa, centralizados entre si.
    X_MEDIA = 12.6
    anos_ytd = [a for a in anos if tem_ytd and not pd.isna(ytd.get(a, np.nan))]
    desloc_ytd = {a: X_MEDIA + (j - (len(anos_ytd) - 1) / 2) * passo
                  for j, a in enumerate(anos_ytd)}

    def rotulo(valor, taxa_valor, casas=0):
        # `formato` e uma funcao (valor, casas) -> texto, para unidades como
        # milhoes; sem ela o rotulo e o proprio numero no padrao brasileiro
        if pd.isna(valor):
            return ""
        texto = formato(valor, casas) if formato else _br(valor, casas)
        if taxa_valor is not None and not pd.isna(taxa_valor):
            texto += f" ({_br(taxa_valor * 100, 1)}%)"
        return texto

    def marca_delta(x, y, atual, anterior):
        # A cor segue o RESULTADO, nao o sinal: em indicadores em que cair e bom,
        # como churn, a queda aparece em verde. O sinal (+/-) continua explicito,
        # entao a leitura nao depende da cor.
        if pd.isna(atual) or pd.isna(anterior) or anterior == 0:
            return
        var = atual / anterior - 1
        bom = (var >= 0) if alta_e_boa else (var <= 0)
        anotacoes.append(dict(
            x=x, y=y, yshift=9,
            text="<b>" + f"{var:+.1%}".replace(".", ",") + "</b>",
            showarrow=False, textangle=-90, xanchor="center", yanchor="bottom",
            font=dict(size=10, color=c["delta_sobe"] if bom else c["delta_cai"])))

    for i, ano in enumerate(anos):
        valores = matriz.loc[ano]
        taxas = taxa.loc[ano] if taxa is not None else None
        fig.add_bar(
            x=[m - 1 + desloc[i] for m in valores.index], y=valores.values,
            name=str(ano), width=largura,
            marker=dict(color=cores[i], cornerradius=4, line=dict(width=0)),
            text=[rotulo(valores[m], taxas[m] if taxas is not None else None)
                  for m in valores.index],
            textposition="inside", insidetextanchor="middle", textangle=-90,
            textfont=dict(size=11, color=tintas[i]), cliponaxis=False,
            customdata=[[MESES_PT[m - 1], ano] for m in valores.index],
            hovertemplate="<b>%{customdata[0]}/%{customdata[1]}</b><br>"
                          "%{y:,.0f}<extra></extra>")

        if ano - 1 in matriz.index:
            anterior = matriz.loc[ano - 1]
            for m in valores.index:
                marca_delta(m - 1 + desloc[i], valores[m], valores[m], anterior[m])

        if tem_ytd and ano in anos_ytd:
            valor_media = ytd.get(ano, np.nan)
            fig.add_bar(
                x=[desloc_ytd[ano]], y=[valor_media], name=str(ano),
                width=largura, showlegend=False,
                marker=dict(color=cores[i], cornerradius=4, line=dict(width=0)),
                text=[rotulo(valor_media,
                             ytd_taxa.get(ano, np.nan) if ytd_taxa is not None else None,
                             casas=1)],
                textposition="inside", insidetextanchor="middle", textangle=-90,
                textfont=dict(size=11, color=tintas[i]), cliponaxis=False,
                hovertemplate=f"<b>{rotulo_ytd} {ano}</b><br>%{{y:,.1f}}<extra></extra>")
            if ano - 1 in ytd.index:
                marca_delta(desloc_ytd[ano], valor_media, valor_media,
                            ytd.get(ano - 1, np.nan))

    maximo = float(np.nanmax(matriz.values))
    ticks = list(range(12)) + ([X_MEDIA] if tem_ytd else [])
    rotulos_x = MESES_PT + ([rotulo_ytd] if tem_ytd else [])
    limite_x = [-0.6, X_MEDIA + 0.65] if tem_ytd else [-0.6, 11.6]

    if tem_ytd:
        # Vao entre os meses e a media, marcado por uma linha tenue
        fig.add_vline(x=(11 + X_MEDIA) / 2, line_width=1, line_color=c["grade"])

    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:12px;color:"
                 f"{c['secundaria']}'>{subtitulo}</span>",
            font=dict(size=17, color=c["tinta"]), x=0, xref="paper", y=0.96),
        barmode="overlay", bargap=0, separators=",.",
        plot_bgcolor=c["superficie"], paper_bgcolor=c["superficie"],
        annotations=anotacoes, height=altura,
        # em versoes baixas do grafico (dashboard) o cabecalho nao pode comer
        # 40% da altura util
        margin=dict(t=124 if altura >= 420 else 96, b=60, l=56, r=24),
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif',
                  color=c["secundaria"], size=12),
        legend=dict(orientation="h", y=1.06, x=0, xanchor="left", yanchor="bottom",
                    title_text="", font=dict(color=c["secundaria"])),
        hoverlabel=dict(bgcolor=c["superficie"], bordercolor=c["eixo"],
                        font=dict(color=c["tinta"])),
        xaxis=dict(tickmode="array", tickvals=ticks, ticktext=rotulos_x,
                   showgrid=False, zeroline=False, linecolor=c["eixo"],
                   tickfont=dict(color=c["suave"]), range=limite_x),
        yaxis=dict(title=None, gridcolor=c["grade"], zeroline=False, showline=False,
                   tickfont=dict(color=c["suave"]), range=[0, maximo * 1.24],
                   tickformat=","))
    return fig


def tabela_apoio(matriz, ytd=None, taxa=None, ytd_taxa=None, rotulo_ytd="Media"):
    """Mesma informacao do grafico em numeros, como visao alternativa."""
    valores = matriz.copy()
    valores.columns = MESES_PT
    if ytd is not None:
        valores[rotulo_ytd] = ytd
    inteiro = lambda v: f"{v:,.0f}" if abs(v - round(v)) < 1e-9 else f"{v:,.1f}"
    saida = valores.to_string(na_rep="-", float_format=inteiro)
    if taxa is not None:
        pct = taxa.copy() * 100
        pct.columns = MESES_PT
        if ytd_taxa is not None:
            pct[rotulo_ytd] = ytd_taxa * 100
        saida += "\n\nSobre a base ativa do mês anterior (%):\n" + pct.round(1).to_string(na_rep="-")
    return saida


# -----------------------------------------------------------------------------
# 4.1 Evolucao da base ativa
# -----------------------------------------------------------------------------
BASE_MENSAL = base_ativa_mensal(df)
matriz_base = matriz_ano_mes(df[df["transacionou"] == 1], "Documento", contagem=True)

fig_base_ativa = grafico_colunas_ano(
    matriz_base, paleta="verde",
    titulo="Evolução da base ativa",
    subtitulo="Clientes com TPV Total maior que zero no mês. "
              "Acima da coluna, a variação contra o mesmo mês do ano anterior.")

print("Base ativa por ano e mês (documentos com TPV Total > 0):")
print(tabela_apoio(matriz_base))

fig_base_ativa

Base ativa por ano e mês (documentos com TPV Total > 0):
       Jan   Fev   Mar   Abr   Mai   Jun   Jul   Ago   Set   Out   Nov   Dez
ano                                                                         
2024 1,359 1,363 1,390 1,374 1,406 1,433 1,437 1,465 1,466 1,469 1,505 1,519
2025 1,534 1,486 1,496 1,469 1,489 1,499 1,507 1,502 1,507 1,523 1,511 1,526
2026 1,527 1,482 1,469 1,483 1,501 1,519 1,559     -     -     -     -     -


### 4.2 Novos ativos

In [22]:
# -----------------------------------------------------------------------------
# 4.2 Novos ativos
# -----------------------------------------------------------------------------
# Dentro da coluna: o total do mes e, entre parenteses, o indicador dividido pela
# base ativa do mes ANTERIOR. O ultimo grupo traz a MEDIA mensal do ano ate o
# ultimo mes fechado do ano mais recente - por ser media, fica na mesma escala
# das colunas mensais e o grafico mantem um unico eixo.
# Jan a mar/2024 nao tem os 3 meses de historico que a regra exige e ficam
# fora do grafico; por isso a media de 2024 tambem nao e calculada.

matriz_novos_ativos = matriz_ano_mes(df, "flag_novo_ativo", desde=df["periodo"].min() + 3)
taxa_novos_ativos = matriz_taxa(matriz_novos_ativos, BASE_MENSAL)
media_novos_ativos, media_taxa_novos_ativos, janela_novos_ativos = media_ytd(matriz_novos_ativos, BASE_MENSAL)
rotulo_novos_ativos = f"Média Jan-{MESES_PT[janela_novos_ativos[-1] - 1]}"

fig_novos_ativos = grafico_colunas_ano(
    matriz_novos_ativos, paleta="azul",
    titulo="Novos ativos",
    subtitulo="Clientes que transacionaram no mês após três meses zerados. Entre parênteses, o percentual sobre a base ativa do mês anterior.<br>O último grupo é a média mensal do ano. Jan a mar/2024 não têm janela de histórico, então 2024 não entra na média.",
    taxa=taxa_novos_ativos, ytd=media_novos_ativos, ytd_taxa=media_taxa_novos_ativos,
    rotulo_ytd=rotulo_novos_ativos, alta_e_boa=True)

print("Novos ativos por ano e mês:")
print(tabela_apoio(matriz_novos_ativos, media_novos_ativos, taxa_novos_ativos, media_taxa_novos_ativos,
                   rotulo_ytd=rotulo_novos_ativos))

fig_novos_ativos

Novos ativos por ano e mês:
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    -   49   69   70   61   72   62   53   73   75              -
2025   83   51   55   49   52   78   68   52   60   68   57   75           62.3
2026   67   63   69   75   75   78  103    -    -    -    -    -           75.7

Sobre a base ativa do mês anterior (%):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    - 3.50 5.00 5.00 4.30 5.00 4.20 3.60 5.00 5.00              -
2025 5.50 3.30 3.70 3.30 3.50 5.20 4.50 3.50 4.00 4.50 3.70 5.00           4.20
2026 4.40 4.10 4.70 5.10 5.10 5.20 6.80    -    -    -    -    -           5.00


### 4.3 Reativacoes

In [23]:
# -----------------------------------------------------------------------------
# 4.3 Reativacoes
# -----------------------------------------------------------------------------
# Dentro da coluna: o total do mes e, entre parenteses, o indicador dividido pela
# base ativa do mes ANTERIOR. O ultimo grupo traz a MEDIA mensal do ano ate o
# ultimo mes fechado do ano mais recente - por ser media, fica na mesma escala
# das colunas mensais e o grafico mantem um unico eixo.
# Jan a mar/2024 nao tem os 3 meses de historico que a regra exige e ficam
# fora do grafico; por isso a media de 2024 tambem nao e calculada.

matriz_reativacoes = matriz_ano_mes(df, "flag_reativacao", desde=df["periodo"].min() + 3)
taxa_reativacoes = matriz_taxa(matriz_reativacoes, BASE_MENSAL)
media_reativacoes, media_taxa_reativacoes, janela_reativacoes = media_ytd(matriz_reativacoes, BASE_MENSAL)
rotulo_reativacoes = f"Média Jan-{MESES_PT[janela_reativacoes[-1] - 1]}"

fig_reativacoes = grafico_colunas_ano(
    matriz_reativacoes, paleta="amarelo",
    titulo="Reativações",
    subtitulo="Clientes que voltaram a transacionar após um mês zerado, com TPV em M-2 ou M-3. Entre parênteses, o percentual sobre a base ativa do mês anterior.<br>O último grupo é a média mensal do ano. Jan a mar/2024 não têm janela de histórico, então 2024 não entra na média.",
    taxa=taxa_reativacoes, ytd=media_reativacoes, ytd_taxa=media_taxa_reativacoes,
    rotulo_ytd=rotulo_reativacoes, alta_e_boa=True)

print("Reativações por ano e mês:")
print(tabela_apoio(matriz_reativacoes, media_reativacoes, taxa_reativacoes, media_taxa_reativacoes,
                   rotulo_ytd=rotulo_reativacoes))

fig_reativacoes

Reativações por ano e mês:
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    -   18   28   29   18   30   29   21   27   17              -
2025   24   27   36   35   33   20   25   19   20   20   18   27           28.6
2026   21   23   32   27   25   24   18    -    -    -    -    -           24.3

Sobre a base ativa do mês anterior (%):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    - 1.30 2.00 2.10 1.30 2.10 2.00 1.40 1.80 1.10              -
2025 1.60 1.80 2.40 2.30 2.20 1.30 1.70 1.30 1.30 1.30 1.20 1.80           1.90
2026 1.40 1.50 2.20 1.80 1.70 1.60 1.20    -    -    -    -    -           1.60


### 4.4 Churn

In [24]:
# -----------------------------------------------------------------------------
# 4.4 Churn
# -----------------------------------------------------------------------------
# Dentro da coluna: o total do mes e, entre parenteses, o indicador dividido pela
# base ativa do mes ANTERIOR. O ultimo grupo traz a MEDIA mensal do ano ate o
# ultimo mes fechado do ano mais recente - por ser media, fica na mesma escala
# das colunas mensais e o grafico mantem um unico eixo.
# Jan/2024 nao tem mes anterior no painel e fica fora do grafico; por isso a
# media de 2024 tambem nao e calculada.

matriz_churn = matriz_ano_mes(df, "flag_churn", desde=df["periodo"].min() + 1)
taxa_churn = matriz_taxa(matriz_churn, BASE_MENSAL)
media_churn, media_taxa_churn, janela_churn = media_ytd(matriz_churn, BASE_MENSAL)
rotulo_churn = f"Média Jan-{MESES_PT[janela_churn[-1] - 1]}"

fig_churn = grafico_colunas_ano(
    matriz_churn, paleta="vermelho",
    titulo="Churn",
    subtitulo="Clientes que zeraram o TPV no mês tendo transacionado no mês anterior. Entre parênteses, o percentual sobre a base ativa do mês anterior.<br>O último grupo é a média mensal do ano. Jan/2024 não tem mês anterior no painel, então 2024 não entra na média. A cor segue o resultado: queda do churn aparece em verde.",
    taxa=taxa_churn, ytd=media_churn, ytd_taxa=media_taxa_churn,
    rotulo_ytd=rotulo_churn, alta_e_boa=False)

print("Churn por ano e mês:")
print(tabela_apoio(matriz_churn, media_churn, taxa_churn, media_taxa_churn,
                   rotulo_ytd=rotulo_churn))

fig_churn

Churn por ano e mês:
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -   76   65   83   65   72   74   74   89   70   64   78              -
2025   92  126   81  110   65   88   85   76   73   72   87   87           92.4
2026   76  128  111   85   74   83   80    -    -    -    -    -             91

Sobre a base ativa do mês anterior (%):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    - 5.60 4.80 6.00 4.70 5.10 5.20 5.10 6.10 4.80 4.40 5.20              -
2025 6.10 8.20 5.50 7.40 4.40 5.90 5.70 5.00 4.90 4.80 5.70 5.80           6.20
2026 5.00 8.40 7.50 5.80 5.00 5.50 5.30    -    -    -    -    -           6.10


### 4.5 Ponte da base ativa em 2026

In [25]:
# -----------------------------------------------------------------------------
# 4.5 Ponte da base ativa mes a mes (waterfall)
# -----------------------------------------------------------------------------
# Consolida os tres indicadores anteriores ao longo de 2026: parte da base ativa
# de dez/25 e, a cada mes, soma novos ativos e reativacoes e desconta o churn,
# fechando na base ativa daquele mes.
#
# A identidade base(m) = base(m-1) + novos + reativacoes - churn nao fecha
# sozinha: alguns cadastros transacionaram no mes anterior e sairam da carteira
# no mes seguinte, sem linha na base para receber a flag de churn. Em vez de
# esconder, entram como etapa propria em cinza neutro.
#
# ESCALA: aqui os fluxos mensais (60 a 130 clientes) sao pequenos diante da base
# (cerca de 1.500). Nenhuma barra parte do zero - toda barra representa uma
# DIFERENCA e seu tamanho e proporcional a essa diferenca. O eixo mostra o nivel
# da base ativa, entao o valor absoluto se le no eixo e nos rotulos de fim de
# mes. Desenhar colunas de base cheias aqui exigiria um eixo de 0 a 1.600, em
# que um fluxo de 100 clientes viraria um risco de poucos pixels.

def etapas_ponte_mensal(dados, ano, base_mensal):
    """Fluxos mes a mes de um ano, mais a base de partida (dez do ano anterior)."""
    meses = [p for p in base_mensal.index if p.year == ano]
    inicio = pd.Period(year=ano - 1, month=12, freq="M")
    fluxos = dados.groupby("periodo")[["flag_novo_ativo", "flag_reativacao",
                                       "flag_churn"]].sum()
    passos = []
    for periodo in meses:
        novos = float(fluxos.loc[periodo, "flag_novo_ativo"])
        reativados = float(fluxos.loc[periodo, "flag_reativacao"])
        perdidos = float(fluxos.loc[periodo, "flag_churn"])
        base_anterior = float(base_mensal[periodo - 1])
        base_atual = float(base_mensal[periodo])
        saidas = base_atual - (base_anterior + novos + reativados - perdidos)
        passos.append(dict(periodo=periodo, base=base_atual, fluxos=[
            ("Novos ativos", novos, VERDE_CLARO),
            ("Reativações", reativados, PALETAS["amarelo"]["light"][1]),
            ("Churn", -perdidos, PALETAS["vermelho"]["light"][1]),
            ("Saídas da carteira", saidas, CHROME["light"]["suave"]),
        ]))
    return inicio, float(base_mensal[inicio]), passos


def grafico_ponte_mensal(inicio, base_inicial, passos, titulo, subtitulo,
                         base_zero=False, modo="light", altura=560):
    """Waterfall continuo: cada barra e um fluxo, o eixo mostra o nivel da base.

    base_zero=True ancora o eixo no zero, mostrando a proporcao real entre os
    fluxos mensais e o tamanho da carteira - ao custo de barras bem menores.
    """
    c = CHROME[modo]
    fig = go.Figure()
    formas, anotacoes = [], []
    corrente = base_inicial

    largura, vao = 0.68, 1.0     # vao de um slot entre os meses
    passo_mes = len(passos[0]["fluxos"]) + vao
    x_inicio = -0.9              # marcador da base de partida

    def marcador_nivel(x0, x1, y):
        """Trace curto com o nivel da base ativa e o numero em destaque."""
        formas.append(dict(type="line", x0=x0, x1=x1, y0=y, y1=y,
                           line=dict(color=c["eixo"], width=1.5)))
        anotacoes.append(dict(
            x=(x0 + x1) / 2, y=y, yshift=10, showarrow=False,
            text=f"<b>{_br(y)}</b>",
            font=dict(size=11, color=c["tinta"]), xanchor="center",
            yanchor="bottom"))

    marcador_nivel(x_inicio - 0.45, x_inicio + 0.45, base_inicial)
    anotacoes.append(dict(
        x=x_inicio, y=base_inicial, yshift=-12, showarrow=False,
        text=f"Base<br>{MESES_PT[inicio.month - 1]}/{str(inicio.year)[2:]}",
        font=dict(size=10, color=c["secundaria"]), xanchor="center", yanchor="top"))

    x_anterior = x_inicio + 0.45
    for i, mes in enumerate(passos):
        base_x = i * passo_mes
        for j, (nome, valor, cor) in enumerate(mes["fluxos"]):
            x = base_x + j
            if valor == 0:
                continue
            fundo = corrente + min(valor, 0.0)
            topo = corrente + valor
            fig.add_bar(
                x=[x], y=[abs(valor)], base=[fundo], width=largura,
                marker=dict(color=cor, cornerradius=3, line=dict(width=0)),
                text=[("+" if valor > 0 else "−") + _br(abs(valor))],
                textposition="outside", textangle=-90, cliponaxis=False,
                textfont=dict(size=10, color=c["tinta"]), showlegend=False,
                customdata=[[nome, MESES_PT[mes["periodo"].month - 1],
                             corrente, topo]],
                hovertemplate="<b>%{customdata[0]} · %{customdata[1]}</b><br>"
                              "de %{customdata[2]:,.0f} para %{customdata[3]:,.0f}"
                              "<extra></extra>")
            # conector do topo anterior ate esta barra
            formas.append(dict(type="line", x0=x_anterior, x1=x - largura / 2,
                               y0=corrente, y1=corrente,
                               line=dict(color=c["grade"], width=1)))
            x_anterior = x + largura / 2
            corrente = topo

        # fecha o mes com o marcador de nivel da base
        fim_x = base_x + len(mes["fluxos"]) - 1 + largura / 2
        formas.append(dict(type="line", x0=x_anterior, x1=fim_x + 0.55,
                           y0=corrente, y1=corrente,
                           line=dict(color=c["grade"], width=1)))
        marcador_nivel(fim_x + 0.15, fim_x + 0.95, corrente)
        x_anterior = fim_x + 0.95

        if i < len(passos) - 1:   # separador entre meses
            formas.append(dict(type="line", x0=base_x + passo_mes - 1,
                               x1=base_x + passo_mes - 1, y0=0, y1=1,
                               yref="paper", line=dict(color=c["grade"], width=1)))

    # extremos do eixo: todos os topos e fundos de barra, com folga para rotulos
    extremos = [base_inicial]
    corrente = base_inicial
    for mes in passos:
        for _, valor, _ in mes["fluxos"]:
            extremos += [corrente, corrente + valor]
            corrente += valor
        extremos.append(corrente)
    folga = (max(extremos) - min(extremos)) * 0.34

    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:12px;color:"
                 f"{c['secundaria']}'>{subtitulo}</span>",
            font=dict(size=17, color=c["tinta"]), x=0, xref="paper", y=0.97),
        shapes=formas, annotations=anotacoes, separators=",.",
        plot_bgcolor=c["superficie"], paper_bgcolor=c["superficie"],
        height=altura, margin=dict(t=124, b=64, l=60, r=24), bargap=0,
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif',
                  color=c["secundaria"], size=12),
        hoverlabel=dict(bgcolor=c["superficie"], bordercolor=c["eixo"],
                        font=dict(color=c["tinta"])),
        xaxis=dict(
            tickmode="array",
            # centro do mes considerando as barras mais o marcador de nivel
            tickvals=[i * passo_mes + (len(p["fluxos"]) - 1) / 2 + 0.4
                      for i, p in enumerate(passos)],
            ticktext=[MESES_PT[p["periodo"].month - 1] for p in passos],
            showgrid=False, zeroline=False, linecolor=c["eixo"],
            tickfont=dict(color=c["tinta"], size=12),
            range=[x_inicio - 1.0, (len(passos) - 1) * passo_mes
                   + len(passos[0]["fluxos"]) + 0.2]),
        yaxis=dict(title=None, gridcolor=c["grade"], zeroline=False,
                   showline=False, tickfont=dict(color=c["suave"]),
                   range=([0, max(extremos) * 1.10] if base_zero
                          else [min(extremos) - folga, max(extremos) + folga]),
                   tickformat=","))

    # legenda manual: as cores identificam o tipo de fluxo, nao uma serie
    for nome, cor in [("Novos ativos", VERDE_CLARO),
                      ("Reativações", PALETAS["amarelo"]["light"][1]),
                      ("Churn", PALETAS["vermelho"]["light"][1]),
                      ("Saídas da carteira", CHROME["light"]["suave"])]:
        fig.add_bar(x=[None], y=[None], name=nome, showlegend=True,
                    marker=dict(color=cor))
    fig.update_layout(legend=dict(orientation="h", y=1.045, x=0, xanchor="left",
                                  yanchor="bottom", title_text="",
                                  font=dict(color=c["secundaria"])))
    return fig


INICIO_2026, BASE_INICIAL_2026, PASSOS_2026 = etapas_ponte_mensal(df, 2026, BASE_MENSAL)
fig_ponte_2026 = grafico_ponte_mensal(
    INICIO_2026, BASE_INICIAL_2026, PASSOS_2026,
    titulo="Ponte da base ativa mês a mês em 2026",
    subtitulo="Cada barra é uma diferença, e o eixo mostra o nível da base ativa. "
              "Os números em destaque são a base ao fim de cada mês.<br>"
              "Saídas da carteira são cadastros que pararam de transacionar e "
              "deixaram a base no mesmo mês, sem linha para receber a flag de churn.")

print("Ponte da base ativa em 2026:")
print(f"  Base {MESES_PT[INICIO_2026.month - 1]}/{str(INICIO_2026.year)[2:]}"
      f"{_br(BASE_INICIAL_2026):>26}")
for _mes in PASSOS_2026:
    _detalhe = "  ".join(
        f"{_nome.split()[0].lower()} {_valor:+.0f}" for _nome, _valor, _ in _mes["fluxos"])
    print(f"  {MESES_PT[_mes['periodo'].month - 1]}/{str(_mes['periodo'].year)[2:]}"
          f"   {_detalhe:<52} base {_br(_mes['base']):>7}")

fig_ponte_2026

Ponte da base ativa em 2026:
  Base Dez/25                     1.526
  Jan/26   novos +67  reativações +21  churn -76  saídas -11    base   1.527
  Fev/26   novos +63  reativações +23  churn -128  saídas -3    base   1.482
  Mar/26   novos +69  reativações +32  churn -111  saídas -3    base   1.469
  Abr/26   novos +75  reativações +27  churn -85  saídas -3     base   1.483
  Mai/26   novos +75  reativações +25  churn -74  saídas -8     base   1.501
  Jun/26   novos +78  reativações +24  churn -83  saídas -1     base   1.519
  Jul/26   novos +103  reativações +18  churn -80  saídas -1    base   1.559


### 4.6 Ponte em janelas de 6 meses

Versao alternativa, no formato de colunas cheias. Em janelas semestrais os fluxos
acumulados ficam em ordem de grandeza proxima da base, o que torna esse formato legivel.

In [26]:
# -----------------------------------------------------------------------------
# 4.6 Ponte da base ativa em saltos de 6 meses
# -----------------------------------------------------------------------------
# Versao alternativa da ponte, no formato de colunas cheias: as barras de base
# partem do zero e os fluxos flutuam entre elas. Em janelas de seis meses os
# fluxos acumulam de 400 a 600 clientes, ordem de grandeza proxima da base, o
# que torna esse formato legivel - o que nao acontece no recorte mensal.
#
# A serie comeca em jul/2024 porque novos ativos e reativacoes so existem a
# partir de abr/2024 (a regra exige tres meses de historico), e as janelas sao
# contadas de tras para frente a partir de jul/2026 para todas terem seis meses
# fechados.

def janelas_de_meses(fim, passo, primeiro_valido):
    """Janelas de `passo` meses, contadas de tras para frente a partir de `fim`."""
    janelas, atual = [], fim
    while atual - (passo - 1) >= primeiro_valido:
        janelas.append((atual - (passo - 1), atual))
        atual = atual - passo
    return list(reversed(janelas))


def etapas_ponte_janelas(dados, base_mensal, janelas):
    """Base inicial e, para cada janela, os quatro fluxos e a base ao final."""
    fluxos = dados.groupby("periodo")[["flag_novo_ativo", "flag_reativacao",
                                       "flag_churn"]].sum()
    base_inicial = float(base_mensal[janelas[0][0] - 1])
    blocos, corrente = [], base_inicial
    for inicio, fim in janelas:
        meses = pd.period_range(inicio, fim, freq="M")
        novos = float(fluxos.loc[meses, "flag_novo_ativo"].sum())
        reativados = float(fluxos.loc[meses, "flag_reativacao"].sum())
        perdidos = float(fluxos.loc[meses, "flag_churn"].sum())
        base_final = float(base_mensal[fim])
        saidas = base_final - (corrente + novos + reativados - perdidos)
        blocos.append(dict(
            rotulo=f"{MESES_PT[inicio.month - 1]}/{str(inicio.year)[2:]} a "
                   f"{MESES_PT[fim.month - 1]}/{str(fim.year)[2:]}",
            base=base_final, fim=fim, fluxos=[
                ("Novos ativos", novos, PALETAS["azul"]["light"][1]),
                ("Reativações", reativados, PALETAS["amarelo"]["light"][1]),
                ("Churn", -perdidos, PALETAS["vermelho"]["light"][1]),
                ("Saídas da carteira", saidas, CHROME["light"]["suave"]),
            ]))
        corrente = base_final
    return base_inicial, blocos


def grafico_ponte_janelas(inicio, base_inicial, blocos, titulo, subtitulo,
                          modo="light", altura=560):
    """Colunas de base partindo do zero, fluxos flutuando entre elas."""
    c = CHROME[modo]
    verde = PALETAS["verde"]["light" if modo == "light" else "dark"][2]
    fig = go.Figure()
    formas, anotacoes = [], []
    corrente = base_inicial
    largura, vao = 0.66, 1.0
    passo_bloco = len(blocos[0]["fluxos"]) + 1 + vao   # fluxos + coluna de base + vao

    def coluna_base(x, valor, rotulo_eixo):
        fig.add_bar(
            x=[x], y=[valor], width=largura,
            marker=dict(color=verde, cornerradius=4, line=dict(width=0)),
            text=[_br(valor)], textposition="inside", insidetextanchor="middle",
            textangle=-90, textfont=dict(size=12, color="#ffffff"),
            cliponaxis=False, showlegend=False,
            customdata=[[rotulo_eixo]],
            hovertemplate="<b>Base ativa %{customdata[0]}</b><br>"
                          "%{y:,.0f} clientes<extra></extra>")
        anotacoes.append(dict(x=x, y=0, yshift=-8, yanchor="top", xanchor="center",
                              showarrow=False, text=rotulo_eixo,
                              font=dict(size=11, color=c["tinta"])))

    x_base = -1.0
    coluna_base(x_base, base_inicial,
                f"{MESES_PT[inicio.month - 1]}/{str(inicio.year)[2:]}")
    x_anterior = x_base + largura / 2

    for i, bloco in enumerate(blocos):
        base_x = i * passo_bloco
        for j, (nome, valor, cor) in enumerate(bloco["fluxos"]):
            x = base_x + j
            if valor == 0:
                continue
            fundo, topo = corrente + min(valor, 0.0), corrente + valor
            fig.add_bar(
                x=[x], y=[abs(valor)], base=[fundo], width=largura,
                marker=dict(color=cor, cornerradius=4, line=dict(width=0)),
                text=[("+" if valor > 0 else "−") + _br(abs(valor))],
                textposition="outside", textangle=-90, cliponaxis=False,
                textfont=dict(size=11, color=c["tinta"]), showlegend=False,
                customdata=[[nome, bloco["rotulo"], corrente, topo]],
                hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>"
                              "de %{customdata[2]:,.0f} para %{customdata[3]:,.0f}"
                              "<extra></extra>")
            formas.append(dict(type="line", x0=x_anterior, x1=x - largura / 2,
                               y0=corrente, y1=corrente,
                               line=dict(color=c["eixo"], width=1)))
            x_anterior = x + largura / 2
            corrente = topo

        x_base_bloco = base_x + len(bloco["fluxos"])
        formas.append(dict(type="line", x0=x_anterior,
                           x1=x_base_bloco - largura / 2, y0=corrente, y1=corrente,
                           line=dict(color=c["eixo"], width=1)))
        coluna_base(x_base_bloco, bloco["base"],
                    f"{MESES_PT[bloco['fim'].month - 1]}/{str(bloco['fim'].year)[2:]}")
        x_anterior = x_base_bloco + largura / 2

        # rotulo da janela, acima do bloco de fluxos
        anotacoes.append(dict(
            x=base_x + (len(bloco["fluxos"]) - 1) / 2, y=1.0, yref="paper",
            yshift=-6, yanchor="top", xanchor="center", showarrow=False,
            text=bloco["rotulo"], font=dict(size=11, color=c["secundaria"])))
        if i < len(blocos) - 1:
            formas.append(dict(type="line", x0=base_x + passo_bloco - 1,
                               x1=base_x + passo_bloco - 1, y0=0, y1=1,
                               yref="paper", line=dict(color=c["grade"], width=1)))

    picos = [base_inicial]
    corrente = base_inicial
    for bloco in blocos:
        for _, valor, _ in bloco["fluxos"]:
            corrente += valor
            picos.append(corrente)
    teto = max(picos)

    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:12px;color:"
                 f"{c['secundaria']}'>{subtitulo}</span>",
            font=dict(size=17, color=c["tinta"]), x=0, xref="paper", y=0.97),
        shapes=formas, annotations=anotacoes, separators=",.",
        plot_bgcolor=c["superficie"], paper_bgcolor=c["superficie"],
        height=altura, margin=dict(t=136, b=76, l=60, r=24), bargap=0,
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif',
                  color=c["secundaria"], size=12),
        hoverlabel=dict(bgcolor=c["superficie"], bordercolor=c["eixo"],
                        font=dict(color=c["tinta"])),
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   linecolor=c["eixo"],
                   range=[x_base - 0.9,
                          (len(blocos) - 1) * passo_bloco
                          + len(blocos[0]["fluxos"]) + 0.9]),
        yaxis=dict(title=None, gridcolor=c["grade"], zeroline=False, showline=False,
                   tickfont=dict(color=c["suave"]), range=[0, teto * 1.16],
                   tickformat=","))

    for nome, cor in [("Base ativa", verde),
                      ("Novos ativos", PALETAS["azul"]["light"][1]),
                      ("Reativações", PALETAS["amarelo"]["light"][1]),
                      ("Churn", PALETAS["vermelho"]["light"][1]),
                      ("Saídas da carteira", CHROME["light"]["suave"])]:
        fig.add_bar(x=[None], y=[None], name=nome, showlegend=True,
                    marker=dict(color=cor))
    fig.update_layout(legend=dict(orientation="h", y=1.10, x=0, xanchor="left",
                                  yanchor="bottom", title_text="",
                                  font=dict(color=c["secundaria"])))
    return fig


JANELAS = janelas_de_meses(fim=df["periodo"].max(), passo=6,
                           primeiro_valido=df["periodo"].min() + 3)
BASE_INICIAL_JANELAS, BLOCOS_JANELAS = etapas_ponte_janelas(df, BASE_MENSAL, JANELAS)

fig_ponte_semestral = grafico_ponte_janelas(
    JANELAS[0][0] - 1, BASE_INICIAL_JANELAS, BLOCOS_JANELAS,
    titulo="Ponte da base ativa em janelas de 6 meses",
    subtitulo="Colunas verdes são a base ativa ao fim de cada janela e partem do zero. "
              "Entre elas, os fluxos acumulados do período.<br>"
              "A série começa em jul/2024 porque novos ativos e reativações exigem "
              "três meses de histórico, e as janelas são fechadas de trás para frente "
              "a partir de jul/2026.")

print("Ponte em janelas de 6 meses:")
print(f"  Base {MESES_PT[(JANELAS[0][0] - 1).month - 1]}/"
      f"{str((JANELAS[0][0] - 1).year)[2:]}{_br(BASE_INICIAL_JANELAS):>28}")
for _bloco in BLOCOS_JANELAS:
    _detalhe = "  ".join(f"{_n.split()[0].lower()} {_v:+.0f}"
                         for _n, _v, _ in _bloco["fluxos"])
    print(f"  {_bloco['rotulo']:<18} {_detalhe:<56} base {_br(_bloco['base']):>7}")

fig_ponte_semestral

Ponte em janelas de 6 meses:
  Base Jul/24                       1.437
  Ago/24 a Jan/25    novos +418  reativações +148  churn -467  saídas -2      base   1.534
  Fev/25 a Jul/25    novos +353  reativações +176  churn -555  saídas -1      base   1.507
  Ago/25 a Jan/26    novos +379  reativações +125  churn -471  saídas -13     base   1.527
  Fev/26 a Jul/26    novos +463  reativações +149  churn -561  saídas -19     base   1.559


### 4.7 Ponte mensal em escala absoluta

A mesma ponte de 4.5 com o eixo ancorado no zero, para guardar a referencia de
tamanho entre o giro mensal e a carteira.

In [27]:
# -----------------------------------------------------------------------------
# 4.7 Ponte mensal em escala absoluta, com as taxas em linha
# -----------------------------------------------------------------------------
# Experimento de design. Cada mes traz os quatro fluxos flutuando e fecha com a
# coluna cheia da base ativa, que desce ate o zero e carrega o proprio numero na
# vertical. Assim se le ao mesmo tempo o tamanho da carteira e o peso de cada
# fluxo sobre ela.
#
# DUAS ESCALAS ANCORADAS: a escala de percentual fica a direita e e amarrada a
# da esquerda - 0% sobre o zero e 10% sobre os 1.000 clientes. A ancoragem e fixa
# (o topo do eixo direito e sempre o topo do esquerdo dividido por 100), entao a
# relacao entre as duas escalas nao muda quando os dados mudam, que e o problema
# classico do eixo duplo. Na pratica as linhas ficam na metade de baixo e os
# fluxos no alto, sem disputa de espaco.
#
# O verde claro e reservado a coluna de base, que e pano de fundo do mes; cada
# fluxo usa a cor do seu indicador, entao novos ativos aparece em azul tanto na
# coluna quanto na linha de percentual. O verde claro fica em 1,50:1 de contraste
# contra a superficie, abaixo do piso de 2:1 de uma cor que carrega significado
# sozinha - o numero sobre a coluna e a compensacao.
COR_TAXA = {"Novos ativos": PALETAS["azul"]["light"][1],
            "Reativações": PALETAS["amarelo"]["light"][1],
            "Churn": PALETAS["vermelho"]["light"][1]}
META_CHURN = 5.0            # meta de churn, em % da base ativa do mes anterior
ANCORA_PCT, ANCORA_CLIENTES = 10.0, 1000.0


def grafico_ponte_com_taxas(inicio, base_inicial, passos, base_mensal, titulo,
                            subtitulo, modo="light", altura=660):
    """Ponte mensal com coluna de base por mes e as taxas de fluxo em linha."""
    c = CHROME[modo]
    fig = go.Figure()
    formas, anotacoes = [], []
    corrente = base_inicial
    # vao zero: o espacamento entre a coluna de base e o mes seguinte fica
    # igual ao espacamento entre os fluxos, e a divisao dos meses passa a
    # ser feita so pela linha vertical
    largura, vao = 0.68, 0.0
    n_fluxos = len(passos[0]["fluxos"])
    passo_mes = n_fluxos + 1 + vao          # fluxos + coluna de base + vao
    x_inicio = -(1 + vao)

    cores_coluna = {"Novos ativos": PALETAS["azul"]["light"][1],
                    "Reativações": PALETAS["amarelo"]["light"][1],
                    "Churn": PALETAS["vermelho"]["light"][1],
                    "Saídas da carteira": CHROME[modo]["suave"]}

    def coluna_base(x, valor, legenda):
        """Coluna cheia da base ativa, com o numero na vertical dentro dela."""
        fig.add_bar(
            x=[x], y=[valor], width=largura, showlegend=False,
            marker=dict(color=VERDE_CLARO, cornerradius=4, line=dict(width=0)),
            # numero ACIMA da coluna: dentro dele disputaria espaco com as
            # linhas de percentual e ficaria preso a largura da barra
            text=[f"<b>{_br(valor)}</b>"], textposition="outside", textangle=-90,
            textfont=dict(size=13, color=c["tinta"]), cliponaxis=False,
            customdata=[[legenda]],
            hovertemplate="<b>Base ativa %{customdata[0]}</b><br>"
                          "%{y:,.0f} clientes<extra></extra>")

    coluna_base(x_inicio, base_inicial,
                f"{MESES_PT[inicio.month - 1]}/{str(inicio.year)[2:]}")
    anotacoes.append(dict(
        x=x_inicio, y=0, yshift=-8, showarrow=False, yanchor="top",
        xanchor="center", text=f"{MESES_PT[inicio.month - 1]}/{str(inicio.year)[2:]}",
        font=dict(size=11, color=c["secundaria"])))

    # ---- fluxos e colunas de base ----
    x_anterior, extremos = x_inicio + largura / 2, [base_inicial]
    for i, mes in enumerate(passos):
        base_x = i * passo_mes
        for j, (nome, valor, _) in enumerate(mes["fluxos"]):
            x = base_x + j
            if valor == 0:
                continue
            fundo, topo = corrente + min(valor, 0.0), corrente + valor
            fig.add_bar(
                x=[x], y=[abs(valor)], base=[fundo], width=largura,
                marker=dict(color=cores_coluna[nome], cornerradius=3,
                            line=dict(width=0)),
                text=["<b>" + ("+" if valor > 0 else "−")
                      + _br(abs(valor)) + "</b>"],
                textposition="outside", textangle=-90, cliponaxis=False,
                textfont=dict(size=10, color=c["tinta"]), showlegend=False,
                customdata=[[nome, MESES_PT[mes["periodo"].month - 1], corrente, topo]],
                hovertemplate="<b>%{customdata[0]} · %{customdata[1]}</b><br>"
                              "de %{customdata[2]:,.0f} para %{customdata[3]:,.0f}"
                              "<extra></extra>")
            formas.append(dict(type="line", x0=x_anterior, x1=x - largura / 2,
                               y0=corrente, y1=corrente,
                               line=dict(color=c["grade"], width=1)))
            x_anterior, extremos = x + largura / 2, extremos + [corrente, topo]
            corrente = topo

        x_base = base_x + n_fluxos
        formas.append(dict(type="line", x0=x_anterior, x1=x_base - largura / 2,
                           y0=corrente, y1=corrente,
                           line=dict(color=c["grade"], width=1)))
        coluna_base(x_base, corrente,
                    f"{MESES_PT[mes['periodo'].month - 1]}/"
                    f"{str(mes['periodo'].year)[2:]}")
        x_anterior, extremos = x_base + largura / 2, extremos + [corrente]

        if i < len(passos) - 1:
            meio_do_vao = base_x + n_fluxos + (1 + vao) / 2
            formas.append(dict(type="line", x0=meio_do_vao, x1=meio_do_vao,
                               y0=0, y1=1, yref="paper",
                               line=dict(color=c["grade"], width=1)))

    # ---- linhas das taxas ----
    centros = [i * passo_mes + (n_fluxos - 1) / 2 for i in range(len(passos))]
    taxas = {}
    for nome in COR_TAXA:
        serie = []
        for mes in passos:
            valor = dict((n, v) for n, v, _ in mes["fluxos"])[nome]
            serie.append(abs(valor) / float(base_mensal[mes["periodo"] - 1]) * 100)
        taxas[nome] = serie
        fig.add_scatter(
            x=centros, y=serie, yaxis="y2", mode="lines+markers",
            name=f"{nome} (%)", line=dict(color=COR_TAXA[nome], width=2),
            marker=dict(size=8, color=COR_TAXA[nome],
                        line=dict(color=c["superficie"], width=2)),
            customdata=[[MESES_PT[m["periodo"].month - 1]] for m in passos],
            hovertemplate=f"<b>{nome} · %{{customdata[0]}}</b><br>"
                          "%{y:.1f}% da base do mês anterior<extra></extra>")

    # rotulo de cada ponto: o mais alto do mes vai acima e os demais abaixo, para
    # os numeros nao colidirem quando duas linhas se cruzam
    for k, centro in enumerate(centros):
        ordem = sorted(COR_TAXA, key=lambda n: taxas[n][k], reverse=True)
        for posicao, nome in enumerate(ordem):
            acima = posicao == 0
            anotacoes.append(dict(
                x=centro, y=taxas[nome][k], yref="y2",
                yshift=13 if acima else -13, showarrow=False, xanchor="center",
                yanchor="bottom" if acima else "top",
                text=f"<b>{_br(taxas[nome][k], 1)}%</b>",
                font=dict(size=10, color=COR_TAXA[nome])))

    teto_topo = max(extremos) * 1.16
    teto_pct = teto_topo * ANCORA_PCT / ANCORA_CLIENTES

    formas.append(dict(type="line", xref="paper", x0=0, x1=1, yref="y2",
                       y0=META_CHURN, y1=META_CHURN,
                       line=dict(color=COR_TAXA["Churn"], width=1.5, dash="dash")))
    anotacoes.append(dict(
        x=0.004, xref="paper", y=META_CHURN, yref="y2", yshift=6, showarrow=False,
        text=f"<b>Meta de churn {_br(META_CHURN, 0)}%</b>", xanchor="left",
        yanchor="bottom", font=dict(size=10, color=COR_TAXA["Churn"])))

    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:12px;color:"
                 f"{c['secundaria']}'>{subtitulo}</span>",
            font=dict(size=17, color=c["tinta"]), x=0, xref="paper", y=0.97),
        shapes=formas, annotations=anotacoes, separators=",.",
        plot_bgcolor=c["superficie"], paper_bgcolor=c["superficie"],
        height=altura, margin=dict(t=132, b=68, l=60, r=64), bargap=0,
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif',
                  color=c["secundaria"], size=12),
        legend=dict(orientation="h", y=1.045, x=0, xanchor="left", yanchor="bottom",
                    title_text="", font=dict(color=c["secundaria"])),
        hoverlabel=dict(bgcolor=c["superficie"], bordercolor=c["eixo"],
                        font=dict(color=c["tinta"])),
        xaxis=dict(
            tickmode="array",
            tickvals=[i * passo_mes + n_fluxos / 2 for i in range(len(passos))],
            ticktext=[MESES_PT[p["periodo"].month - 1] for p in passos],
            showgrid=False, zeroline=False, linecolor=c["eixo"],
            tickfont=dict(color=c["tinta"], size=12),
            range=[x_inicio - 0.9,
                   (len(passos) - 1) * passo_mes + n_fluxos + 0.9]),
        yaxis=dict(title=None, gridcolor=c["grade"], zeroline=False,
                   showline=False, tickfont=dict(color=c["suave"]),
                   range=[0, teto_topo], tickformat=","),
        yaxis2=dict(title=None, overlaying="y", side="right", showgrid=False,
                    zeroline=False, showline=False, range=[0, teto_pct],
                    tickmode="array", tickvals=[0, 2, 4, 6, 8, 10],
                    ticktext=["0%", "2%", "4%", "6%", "8%", "10%"],
                    tickfont=dict(color=c["suave"])))

    # legenda das colunas: a cor identifica o tipo de fluxo, nao uma serie
    for nome, cor in [("Base ativa", VERDE_CLARO),
                      ("Novos ativos", PALETAS["azul"]["light"][1]),
                      ("Reativações", PALETAS["amarelo"]["light"][1]),
                      ("Churn", PALETAS["vermelho"]["light"][1]),
                      ("Saídas da carteira", CHROME[modo]["suave"])]:
        fig.add_bar(x=[None], y=[None], name=nome, showlegend=True,
                    marker=dict(color=cor))
    return fig


fig_ponte_2026_absoluta = grafico_ponte_com_taxas(
    INICIO_2026, BASE_INICIAL_2026, PASSOS_2026, BASE_MENSAL,
    titulo="Ponte da base ativa em 2026, com as taxas de fluxo",
    subtitulo="Colunas em escala absoluta, em clientes, no eixo da esquerda. As linhas "
              "trazem cada fluxo como percentual da base ativa do mês anterior,<br>"
              "no eixo da direita, ancorado ao da esquerda: 0% sobre o zero e 10% sobre "
              "os 1.000 clientes. A tracejada é a meta de churn de 5%.")

print("Taxas sobre a base ativa do mês anterior (%):")
for _mes in PASSOS_2026:
    _anterior = float(BASE_MENSAL[_mes["periodo"] - 1])
    _valores = dict((n, v) for n, v, _ in _mes["fluxos"])
    _linha = "   ".join(f"{_n.split()[0].lower()} {abs(_valores[_n]) / _anterior * 100:4.1f}%"
                        for _n in COR_TAXA)
    print(f"  {MESES_PT[_mes['periodo'].month - 1]}/{str(_mes['periodo'].year)[2:]}"
          f"   {_linha}   base {_br(_mes['base']):>7}")

fig_ponte_2026_absoluta

Taxas sobre a base ativa do mês anterior (%):
  Jan/26   novos  4.4%   reativações  1.4%   churn  5.0%   base   1.527
  Fev/26   novos  4.1%   reativações  1.5%   churn  8.4%   base   1.482
  Mar/26   novos  4.7%   reativações  2.2%   churn  7.5%   base   1.469
  Abr/26   novos  5.1%   reativações  1.8%   churn  5.8%   base   1.483
  Mai/26   novos  5.1%   reativações  1.7%   churn  5.0%   base   1.501
  Jun/26   novos  5.2%   reativações  1.6%   churn  5.5%   base   1.519
  Jul/26   novos  6.8%   reativações  1.2%   churn  5.3%   base   1.559


## 5. Analises de TPV

### 5.1 Evolucao mensal do TPV Total

In [28]:
# =============================================================================
# 5. ANALISES DE TPV
# =============================================================================
# 5.1 Evolucao mensal do TPV Total
# -----------------------------------------------------------------------------
# Mesmo formato dos indicadores de carteira: colunas agrupadas por mes, uma por
# ano, e a media mensal do ano fechando o grafico. Aqui o valor e monetario,
# entao a matriz e levada para MILHOES de reais - assim o eixo e os rotulos ficam
# legiveis e a variacao percentual, que e razao, nao muda.
#
# Paleta verde da bandeira do Brasil (#009c3b no passo do meio), validada como
# rampa ordinal: passo mais claro em 2,08:1 de contraste contra a superficie
# (piso 2,0) e passos vizinhos a dE 15,1 e 24,8 em visao normal, 14,2 e 23,4 sob
# simulacao de daltonismo (pisos 15 e 8).

MILHAO = 1e6


def em_milhoes(valor, casas=0):
    """Rotulo monetario compacto: 50 milhoes vira '50MM'."""
    return f"{_br(valor, casas)}MM"


matriz_tpv = matriz_ano_mes(df, "tpv_total") / MILHAO
media_tpv, _, janela_tpv = media_ytd(matriz_tpv)
rotulo_tpv = f"Média Jan-{MESES_PT[janela_tpv[-1] - 1]}"

fig_tpv_total = grafico_colunas_ano(
    matriz_tpv, paleta="verde_bandeira",
    titulo="TPV Total por mês",
    subtitulo="Adquirência mais PIX QR Code validado, em milhões de reais. "
              "Acima da coluna, a variação contra o mesmo mês do ano anterior.<br>"
              "O último grupo é o TPV Total médio mensal do ano, na mesma janela "
              "para todos os anos.",
    ytd=media_tpv, rotulo_ytd=rotulo_tpv, formato=em_milhoes)

print("TPV Total por ano e mês (R$ milhões):")
print(tabela_apoio(matriz_tpv, media_tpv, rotulo_ytd=rotulo_tpv))

print("\nVariação contra o mesmo mês do ano anterior (%):")
_variacao = (matriz_tpv.pct_change() * 100).round(1)
_variacao[rotulo_tpv] = (media_tpv.pct_change() * 100).round(1)
_variacao.columns = MESES_PT + [rotulo_tpv]
print(_variacao.to_string(na_rep="-"))

fig_tpv_total

TPV Total por ano e mês (R$ milhões):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024 37.3 28.1 29.0 27.7 31.9 33.3 45.0 39.2 35.0 38.2 41.2 50.1           33.2
2025 47.0 31.2 36.2 35.2 38.4 37.4 51.2 44.4 39.4 43.4 45.4 55.1           39.5
2026 54.0 37.4 36.3 38.4 43.7 43.5 60.0    -    -    -    -    -           44.8

Variação contra o mesmo mês do ano anterior (%):
       Jan   Fev   Mar   Abr   Mai   Jun   Jul   Ago   Set   Out   Nov   Dez  Média Jan-Jul
ano                                                                                        
2024     -     -     -     -     -     -     -     -     -     -     -     -              -
2025 26.10 11.20 24.80 26.80 20.30 12.50 13.70 13.00 12.80 13.70 10.30 10.00          19.10
2026 15.00 19.80  0.30  9.10 13.90 16.20 17.40     -     -     -     -     -          13.30


### 5.2 TPV medio por cliente ativo

In [29]:
# -----------------------------------------------------------------------------
# 5.2 TPV medio por cliente ativo
# -----------------------------------------------------------------------------
# TPV medio = TPV Total do mes / base ativa do mes. Enquanto o grafico anterior
# mostra o tamanho da operacao, este isola a INTENSIDADE: quanto cada cliente
# que transacionou movimentou, em media. Os dois juntos separam crescimento por
# volume de clientes de crescimento por ticket.
#
# Esta metrica nao entra na celula de feature engineering porque nao e um
# atributo de linha: e razao entre dois agregados do mes. No nivel da linha, TPV
# dividido por um cliente seria o proprio TPV.
#
# Rotulos em MILHARES de reais (25 mil = 25k), entao a matriz vai para o grafico
# ja dividida por 1.000 e o eixo acompanha a mesma unidade.

MIL = 1e3


def em_milhares(valor, casas=0):
    """Rotulo monetario compacto: 25 mil reais vira '25k'."""
    return f"{_br(valor, casas)}k"


matriz_tpv_medio = (matriz_tpv * MILHAO / matriz_base) / MIL
media_tpv_medio, _, janela_tpv_medio = media_ytd(matriz_tpv_medio)
rotulo_tpv_medio = f"Média Jan-{MESES_PT[janela_tpv_medio[-1] - 1]}"

fig_tpv_medio = grafico_colunas_ano(
    matriz_tpv_medio, paleta="verde_bandeira",
    titulo="TPV médio por cliente ativo",
    subtitulo="TPV Total do mês dividido pela base ativa do mês, em milhares de reais. "
              "Acima da coluna, a variação contra o mesmo mês do ano anterior.<br>"
              "O último grupo é a média mensal do ano, na mesma janela para todos os anos.",
    ytd=media_tpv_medio, rotulo_ytd=rotulo_tpv_medio, formato=em_milhares)

print("TPV médio por cliente ativo (R$ mil):")
print(tabela_apoio(matriz_tpv_medio, media_tpv_medio, rotulo_ytd=rotulo_tpv_medio))

print("\nDecomposição do crescimento na janela Jan-Jul (%):")
_base_media, _, _ = media_ytd(matriz_base)
_decomposicao = pd.DataFrame({
    "TPV Total": media_tpv.pct_change() * 100,
    "Base ativa": _base_media.pct_change() * 100,
    "TPV médio": media_tpv_medio.pct_change() * 100})
print(_decomposicao.round(1).to_string(na_rep="-"))

fig_tpv_medio

TPV médio por cliente ativo (R$ mil):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024 27.4 20.6 20.9 20.2 22.7 23.2 31.3 26.8 23.8 26.0 27.3 33.0           23.8
2025 30.6 21.0 24.2 23.9 25.8 25.0 33.9 29.5 26.2 28.5 30.0 36.1           26.4
2026 35.4 25.2 24.7 25.9 29.1 28.7 38.5    -    -    -    -    -           29.6

Decomposição do crescimento na janela Jan-Jul (%):
      TPV Total  Base ativa  TPV médio
ano                                   
2024          -           -          -
2025      19.10        7.40      10.90
2026      13.30        0.60      12.50


### 5.3 TPV do Novo Ativo no mes seguinte

In [30]:
# -----------------------------------------------------------------------------
# 5.3 TPV do Novo Ativo no mes seguinte (TPV NA M1)
# -----------------------------------------------------------------------------
# Mede a QUALIDADE da venda nova: quanto o cliente que entrou no mes anterior
# transacionou no seu primeiro mes cheio. Contar novos ativos diz se a operacao
# vende; isto diz se a operacao vende BEM.
#
# Os tres graficos desta secao usam valores entre 0,3 e 2,5 milhoes, entao o
# rotulo sai sempre com uma casa decimal - com numero inteiro, 1,4 e 1,5 milhoes
# virariam o mesmo "1MM".

def em_milhoes_1(valor, casas=None):
    """Rotulo monetario em milhoes, sempre com uma casa decimal."""
    return f"{_br(valor, 1)}MM"


def matriz_tpv_por_flag(dados, coluna_valor, coluna_flag, desde=None):
    """Soma de `coluna_valor` nas linhas em que `coluna_flag` vale 1, em milhoes."""
    recorte = dados[dados[coluna_flag] == 1]
    return matriz_ano_mes(recorte, coluna_valor, desde=desde) / MILHAO


# a flag so existe a partir de mai/2024: Novo Ativo comeca em abr/2024 e esta
# feature olha para o mes anterior
matriz_tpv_na_m1 = matriz_tpv_por_flag(
    df, "tpv_total", "flag_novo_ativo_m1", desde=df["periodo"].min() + 4)
media_tpv_na_m1, _, janela_na = media_ytd(matriz_tpv_na_m1)
rotulo_na = f"Média Jan-{MESES_PT[janela_na[-1] - 1]}"

fig_tpv_na_m1 = grafico_colunas_ano(
    matriz_tpv_na_m1, paleta="azul",
    titulo="TPV do Novo Ativo no mês seguinte",
    subtitulo="TPV Total do mês dos clientes que foram Novo Ativo no mês anterior, "
              "em milhões de reais.<br>"
              "Mede o tamanho da venda nova, não a quantidade. "
              "A série começa em mai/2024, quando a flag do mês anterior passa a existir.",
    ytd=media_tpv_na_m1, rotulo_ytd=rotulo_na, formato=em_milhoes_1)

print("TPV do Novo Ativo no mês seguinte (R$ milhões):")
print(tabela_apoio(matriz_tpv_na_m1, media_tpv_na_m1, rotulo_ytd=rotulo_na))

fig_tpv_na_m1

TPV do Novo Ativo no mês seguinte (R$ milhões):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    -    -  1.4  1.1  2.1  1.1  1.4  1.0  1.3  2.0              -
2025  1.9  1.3  1.0  1.1  1.6  0.9  1.6  1.9  1.4  1.5  1.4  2.5            1.3
2026  2.5  1.6  2.0  1.4  1.8  1.7  2.3    -    -    -    -    -            1.9


### 5.4 Perda de TPV por churn, media dos 3 meses anteriores

In [31]:
# -----------------------------------------------------------------------------
# 5.4 Perda de TPV por churn, pela media dos 3 meses anteriores
# -----------------------------------------------------------------------------
# Mede o tamanho do cliente perdido: para cada cliente em churn no mes, a media
# do seu TPV Total em M-1, M-2 e M-3. A janela de tres meses e deliberada -
# quem sai costuma reduzir o transacional antes de parar, entao olhar so o
# ultimo mes subestima o que a carteira perdeu.
#
# Serie a partir de abr/2024, quando os tres lags passam a existir.

INICIO_PERDA = df["periodo"].min() + 3

matriz_perda_m1_m3 = matriz_tpv_por_flag(
    df, "tpv_total_medio_m1_m3", "flag_churn", desde=INICIO_PERDA)
media_perda_m1_m3, _, janela_perda = media_ytd(matriz_perda_m1_m3)
rotulo_perda = f"Média Jan-{MESES_PT[janela_perda[-1] - 1]}"

fig_perda_churn_m1_m3 = grafico_colunas_ano(
    matriz_perda_m1_m3, paleta="vermelho",
    titulo="Perda de TPV por churn, média dos 3 meses anteriores",
    subtitulo="Para cada cliente em churn no mês, a média do seu TPV Total em M-1, M-2 "
              "e M-3, somada, em milhões de reais.<br>"
              "A janela de três meses captura o tamanho do cliente antes da queda que "
              "antecede a saída. Queda desta série é resultado bom.",
    ytd=media_perda_m1_m3, rotulo_ytd=rotulo_perda, alta_e_boa=False,
    formato=em_milhoes_1)

print("Perda de TPV por churn, média M1-M3 (R$ milhões):")
print(tabela_apoio(matriz_perda_m1_m3, media_perda_m1_m3, rotulo_ytd=rotulo_perda))

fig_perda_churn_m1_m3

Perda de TPV por churn, média M1-M3 (R$ milhões):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    -  0.9  0.9  0.7  0.7  0.5  0.7  0.5  0.7  0.9              -
2025  0.9  2.7  0.9  1.6  0.5  0.9  0.9  0.8  1.2  0.8  1.4  1.0            1.2
2026  1.6  2.0  1.7  0.7  0.9  1.1  1.0    -    -    -    -    -            1.3


### 5.5 Perda de TPV por churn, apenas pelo mes anterior

In [32]:
# -----------------------------------------------------------------------------
# 5.5 Perda de TPV por churn, apenas pelo mes anterior
# -----------------------------------------------------------------------------
# Mesma populacao de 5.4 com a definicao mais simples: o TPV Total do cliente em
# M-1, o ultimo mes em que ele transacionou. Fica sistematicamente menor que a
# media de tres meses, e a diferenca entre as duas series e justamente a queda
# que o cliente ja tinha acumulado antes de sair.
#
# Mesma janela de 5.4 (a partir de abr/2024) para as duas serem comparaveis.

matriz_perda_m1 = matriz_tpv_por_flag(
    df, "tpv_total_m1", "flag_churn", desde=INICIO_PERDA)
media_perda_m1, _, _ = media_ytd(matriz_perda_m1)

fig_perda_churn_m1 = grafico_colunas_ano(
    matriz_perda_m1, paleta="vermelho",
    titulo="Perda de TPV por churn, apenas pelo mês anterior",
    subtitulo="Para cada cliente em churn no mês, o seu TPV Total em M-1, somado, em "
              "milhões de reais.<br>"
              "Definição mais conservadora: só o último mês transacionado. "
              "Queda desta série é resultado bom.",
    ytd=media_perda_m1, rotulo_ytd=rotulo_perda, alta_e_boa=False,
    formato=em_milhoes_1)

print("Perda de TPV por churn, apenas M1 (R$ milhões):")
print(tabela_apoio(matriz_perda_m1, media_perda_m1, rotulo_ytd=rotulo_perda))

print("\nComparação das duas definições de perda (R$ milhões e razão):")
_comparacao = pd.DataFrame({
    "media M1-M3": media_perda_m1_m3,
    "apenas M1": media_perda_m1,
    "M1 / M1-M3": media_perda_m1 / media_perda_m1_m3})
print(_comparacao.round(2).to_string(na_rep="-"))

_quedas = (matriz_perda_m1 / matriz_perda_m1_m3).stack()
print(f"\nEm {(_quedas < 1).mean():.0%} dos meses o TPV de M-1 é menor que a média "
      f"de 3 meses. Razão mediana: {_quedas.median():.2f}.")

fig_perda_churn_m1

Perda de TPV por churn, apenas M1 (R$ milhões):
      Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez  Média Jan-Jul
ano                                                                            
2024    -    -    -  0.4  0.7  0.4  0.3  0.5  0.8  0.3  0.4  0.5              -
2025  0.7  2.0  0.3  1.0  0.4  0.6  0.4  0.4  0.8  0.4  1.0  0.7            0.8
2026  1.0  1.6  0.9  0.4  0.6  1.1  0.8    -    -    -    -    -            0.9

Comparação das duas definições de perda (R$ milhões e razão):
      media M1-M3  apenas M1  M1 / M1-M3
ano                                     
2024            -          -           -
2025         1.20       0.78        0.65
2026         1.26       0.91        0.72

Em 72% dos meses o TPV de M-1 é menor que a média de 3 meses. Razão mediana: 0.66.


### 5.6 Ponte do TPV por safra de cliente, janela Jan-Jul

In [33]:
# -----------------------------------------------------------------------------
# 5.6 Ponte do TPV por safra de cliente, janela Jan-Jul de 2025 para 2026
# -----------------------------------------------------------------------------
# Decompoe a variacao do TPV medio mensal entre duas janelas, classificando cada
# cliente pela sua situacao nas duas pontas:
#   entrou  -> tem TPV na janela nova e nao tinha na anterior
#   ficou   -> tem TPV nas duas, e contribui com a DIFERENCA
#   saiu    -> tinha TPV na janela anterior e nao tem na nova
# A soma fecha por construcao, sem residuo.
#
# POR QUE ANUAL E NAO MES A MES: a ponte mensal fecha, mas o TPV e muito sazonal
# e a barra da base estavel engole as demais - em fev/2026 ela vale -16,3 MM
# contra +1,2 MM de novos ativos, e em jul/2026 vale +15,0 MM. Comparar janelas
# iguais de anos diferentes remove a sazonalidade e deixa os quatro componentes
# na mesma ordem de grandeza.
#
# POR QUE A PERDA NAO PRECISA DA MEDIA DE 3 MESES: nesta janela a barra de saida
# ja e a media mensal do cliente ao longo de sete meses, e nao o TPV do ultimo
# mes. O conflito entre "tamanho do cliente" e "aritmetica que fecha", que existe
# na ponte mensal, nao aparece aqui.
#
# A base estavel e aberta em duas: quem entrou na carteira nos ultimos 12 meses e
# esta em rampa, e a base madura. E a leitura de "TPV M1" dentro da ponte.

def janela_meses(ano, ate_mes):
    return [pd.Period(year=ano, month=m, freq="M") for m in range(1, ate_mes + 1)]


def _rotulo_janela(meses):
    """Janela de um mes vira 'TPV Jul/25'; janela maior vira 'TPV medio 2025'."""
    if len(meses) == 1:
        return f"TPV<br>{MESES_PT[meses[0].month - 1]}/{str(meses[0].year)[2:]}"
    return f"TPV médio<br>{meses[0].year}"


def ponte_tpv_safra(dados, meses_antes, meses_depois, safra_desde, modo="light"):
    """Etapas (rotulo, valor, tipo, cor) da ponte de TPV entre duas janelas."""
    def media_mensal(meses):
        recorte = dados[dados["periodo"].isin(meses)]
        return recorte.groupby("Documento")["tpv_total"].sum() / len(meses)

    antes, depois = media_mensal(meses_antes), media_mensal(meses_depois)
    documentos = antes.index.union(depois.index)
    va = antes.reindex(documentos).fillna(0.0)
    vb = depois.reindex(documentos).fillna(0.0)

    entrou, saiu = (va <= 0) & (vb > 0), (va > 0) & (vb <= 0)
    ficou = (va > 0) & (vb > 0)

    # safra recente: primeira ativacao registrada no painel dentro da janela movel
    primeira = dados.loc[dados["flag_novo_ativo"] == 1].groupby("Documento")["periodo"].min()
    recente = pd.Series(
        [primeira.get(d, pd.Period("1900-01", "M")) >= safra_desde for d in documentos],
        index=documentos)

    # tudo em MILHOES de reais, para o eixo e os rotulos ficarem legiveis
    va, vb = va / MILHAO, vb / MILHAO
    delta = vb - va
    cores = PALETAS["verde_bandeira"][modo]
    etapas = [
        (_rotulo_janela(meses_antes), float(va.sum()), "total", cores[2]),
        ("Entradas", float(vb[entrou].sum()), "fluxo", PALETAS["azul"][modo][1]),
        ("Safra recente", float(delta[ficou & recente].sum()), "fluxo",
         PALETAS["azul"][modo][0]),
        ("Base madura", float(delta[ficou & ~recente].sum()), "fluxo", cores[0]),
        ("Saídas", -float(va[saiu].sum()), "fluxo", PALETAS["vermelho"][modo][1]),
        (_rotulo_janela(meses_depois), float(vb.sum()), "total", cores[2]),
    ]
    contagem = {"Entradas": int(entrou.sum()), "Safra recente": int((ficou & recente).sum()),
                "Base madura": int((ficou & ~recente).sum()), "Saídas": int(saiu.sum())}
    return etapas, contagem


def grafico_ponte_tpv(etapas, titulo, subtitulo, modo="light", altura=540):
    """Waterfall de colunas cheias: totais partem do zero, fluxos flutuam entre eles.

    Aqui os fluxos valem de 1 a 11 milhoes contra uma base de 40, ordem de
    grandeza proxima o bastante para as colunas cheias funcionarem sem distorcer.
    """
    c = CHROME[modo]
    fig = go.Figure()
    formas, corrente = [], 0.0
    rotulos = [e[0] for e in etapas]

    for i, (nome, valor, tipo, cor) in enumerate(etapas):
        if tipo == "total":
            fundo, altura_barra, topo = 0.0, valor, valor
            texto, tinta, posicao = f"<b>{em_milhoes_1(valor)}</b>", "#ffffff", "inside"
        else:
            fundo = corrente + min(valor, 0.0)
            altura_barra, topo = abs(valor), corrente + valor
            sinal = "+" if valor > 0 else "−"
            texto = f"<b>{sinal}{em_milhoes_1(abs(valor))}</b>"
            tinta, posicao = c["tinta"], "outside"

        fig.add_bar(
            x=[rotulos[i]], y=[altura_barra], base=[fundo], width=0.5,
            marker=dict(color=cor, cornerradius=4, line=dict(width=0)),
            text=[texto], textposition=posicao, insidetextanchor="middle",
            textangle=-90 if tipo == "total" else 0,
            textfont=dict(size=13, color=tinta), cliponaxis=False, showlegend=False,
            customdata=[[nome.replace("<br>", " "), corrente, topo]],
            hovertemplate="<b>%{customdata[0]}</b><br>"
                          "de %{customdata[1]:,.0f} para %{customdata[2]:,.0f}"
                          "<extra></extra>")

        if i < len(etapas) - 1:
            formas.append(dict(type="line", x0=i + 0.25, x1=i + 1 - 0.25,
                               y0=topo, y1=topo,
                               line=dict(color=c["eixo"], width=1)))
        corrente = topo

    pico, corrente = 0.0, 0.0
    for _, valor, tipo, _ in etapas:
        corrente = valor if tipo == "total" else corrente + valor
        pico = max(pico, corrente)

    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:12px;color:"
                 f"{c['secundaria']}'>{subtitulo}</span>",
            font=dict(size=17, color=c["tinta"]), x=0, xref="paper", y=0.96),
        shapes=formas, separators=",.", plot_bgcolor=c["superficie"],
        paper_bgcolor=c["superficie"], height=altura, bargap=0.3,
        margin=dict(t=116, b=64, l=60, r=24),
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif',
                  color=c["secundaria"], size=12),
        hoverlabel=dict(bgcolor=c["superficie"], bordercolor=c["eixo"],
                        font=dict(color=c["tinta"])),
        xaxis=dict(showgrid=False, zeroline=False, linecolor=c["eixo"],
                   tickfont=dict(color=c["tinta"], size=12)),
        yaxis=dict(title=None, gridcolor=c["grade"], zeroline=False, showline=False,
                   tickfont=dict(color=c["suave"]), range=[0, pico * 1.14],
                   tickformat=","))
    return fig


ULTIMO_MES = int(df["periodo"].max().month)
ANO_ATUAL = int(df["periodo"].max().year)
SAFRA_DESDE = df["periodo"].max() - 11        # entrou nos ultimos 12 meses

ETAPAS_TPV_ANO, CONTAGEM_TPV_ANO = ponte_tpv_safra(
    df, janela_meses(ANO_ATUAL - 1, ULTIMO_MES), janela_meses(ANO_ATUAL, ULTIMO_MES),
    safra_desde=SAFRA_DESDE)

fig_ponte_tpv_ano = grafico_ponte_tpv(
    ETAPAS_TPV_ANO,
    titulo=f"Ponte do TPV médio mensal, Jan-{MESES_PT[ULTIMO_MES - 1]} de "
           f"{ANO_ATUAL - 1} para {ANO_ATUAL}",
    subtitulo="Cada cliente entra pela sua situação nas duas janelas: entrou, ficou "
              "ou saiu. Valores em milhões de reais por mês.<br>"
              "A base que ficou é aberta entre safra recente, quem entrou nos últimos "
              "12 meses, e base madura. A soma fecha sem resíduo.")

print(f"Ponte do TPV médio mensal, Jan-{MESES_PT[ULTIMO_MES - 1]} "
      f"{ANO_ATUAL - 1} para {ANO_ATUAL}:")
_corrente = 0.0
for _nome, _valor, _tipo, _ in ETAPAS_TPV_ANO:
    _corrente = _valor if _tipo == "total" else _corrente + _valor
    _limpo = _nome.replace("<br>", " ")
    _clientes = CONTAGEM_TPV_ANO.get(_limpo)
    print(f"  {_limpo:<22} {em_milhoes_1(_valor):>8}   acumulado {em_milhoes_1(_corrente):>8}"
          f"   {(str(_clientes) + ' clientes') if _clientes else ''}")

fig_ponte_tpv_ano

Ponte do TPV médio mensal, Jan-Jul 2025 para 2026:
  TPV médio 2025           39,5MM   acumulado   39,5MM   
  Entradas                 10,7MM   acumulado   50,2MM   644 clientes
  Safra recente             0,3MM   acumulado   50,5MM   55 clientes
  Base madura               0,8MM   acumulado   51,3MM   1281 clientes
  Saídas                   -6,5MM   acumulado   44,8MM   552 clientes
  TPV médio 2026           44,8MM   acumulado   44,8MM   


### 5.7 Ponte do TPV do mes contra o mesmo mes do ano anterior

In [34]:
# -----------------------------------------------------------------------------
# 5.7 Ponte do TPV do mes contra o mesmo mes do ano anterior
# -----------------------------------------------------------------------------
# Mesma decomposicao de 5.6 com janela de UM mes: jul/2026 contra jul/2025.
# Comparar o mesmo mes de dois anos tambem neutraliza a sazonalidade, e mantem a
# granularidade mensal que a janela Jan-Jul perde. Num dashboard, o mes vira um
# seletor e este grafico responde "de onde veio a variacao deste mes".
#
# A diferenca de leitura entre 5.6 e 5.7: a janela de sete meses classifica o
# cliente pelo comportamento no periodo todo, entao quem transacionou em qualquer
# mes conta como presente; a janela de um mes e mais dura, e quem simplesmente
# nao transacionou naquele mes ja aparece como saida. Por isso as barras de
# entrada e saida sao maiores aqui.

MES_PONTE = df["periodo"].max()

ETAPAS_TPV_MES, CONTAGEM_TPV_MES = ponte_tpv_safra(
    df, [MES_PONTE - 12], [MES_PONTE], safra_desde=SAFRA_DESDE)

fig_ponte_tpv_mes = grafico_ponte_tpv(
    ETAPAS_TPV_MES,
    titulo=f"Ponte do TPV de {MESES_PT[MES_PONTE.month - 1]}/{str(MES_PONTE.year)[2:]} "
           f"contra {MESES_PT[MES_PONTE.month - 1]}/{str((MES_PONTE - 12).year)[2:]}",
    subtitulo="Mesma decomposição por safra de cliente, com janela de um mês. "
              "Valores em milhões de reais.<br>"
              "Comparar o mesmo mês de dois anos neutraliza a sazonalidade sem abrir "
              "mão da granularidade mensal.")

print(f"Ponte do TPV, {MESES_PT[MES_PONTE.month - 1]}/{str((MES_PONTE - 12).year)[2:]} "
      f"para {MESES_PT[MES_PONTE.month - 1]}/{str(MES_PONTE.year)[2:]}:")
_corrente = 0.0
for _nome, _valor, _tipo, _ in ETAPAS_TPV_MES:
    _corrente = _valor if _tipo == "total" else _corrente + _valor
    _limpo = _nome.replace("<br>", " ")
    _clientes = CONTAGEM_TPV_MES.get(_limpo)
    print(f"  {_limpo:<22} {em_milhoes_1(_valor):>8}   acumulado {em_milhoes_1(_corrente):>8}"
          f"   {(str(_clientes) + ' clientes') if _clientes else ''}")

print("\nAs duas janelas lado a lado (R$ milhões por mês):")
# os rotulos dos totais mudam entre as duas janelas, entao a tabela usa nomes fixos
_etapas_nome = ["TPV inicial", "Entradas", "Safra recente", "Base madura", "Saídas",
                "TPV final"]
_comparacao = pd.DataFrame(
    {"Jan-Jul": [v for _, v, _, _ in ETAPAS_TPV_ANO],
     f"{MESES_PT[MES_PONTE.month - 1]} x {MESES_PT[MES_PONTE.month - 1]}":
        [v for _, v, _, _ in ETAPAS_TPV_MES]}, index=_etapas_nome)
print(_comparacao.round(1).to_string(na_rep="-"))

fig_ponte_tpv_mes

Ponte do TPV, Jul/25 para Jul/26:
  TPV Jul/25               51,2MM   acumulado   51,2MM   
  Entradas                 20,3MM   acumulado   71,4MM   547 clientes
  Safra recente             0,0MM   acumulado   71,5MM   18 clientes
  Base madura               0,2MM   acumulado   71,6MM   994 clientes
  Saídas                  -11,6MM   acumulado   60,0MM   495 clientes
  TPV Jul/26               60,0MM   acumulado   60,0MM   

As duas janelas lado a lado (R$ milhões por mês):
               Jan-Jul  Jul x Jul
TPV inicial      39.50      51.20
Entradas         10.70      20.30
Safra recente     0.30       0.00
Base madura       0.80       0.20
Saídas           -6.50     -11.60
TPV final        44.80      60.00
